# Original

In [ ]:
import os
import re
import numpy as np
import matplotlib.pyplot as plt
import glob
from matplotlib.colors import ListedColormap
from matplotlib.cm import get_cmap
import pandas as pd

def parse_eigenvalue_file(filename):
    """
    Parse eigenvalue files from MLegS output.
    
    Parameters:
    -----------
    filename : str
        Path to the eigenvalue file
        
    Returns:
    --------
    dict
        Dictionary containing the parsed data
    """
    # Extract parameters from filename
    # Update pattern to handle more number formats
    pattern1 = r"bsnsq_eig_bv_([0-9.-]+)_w_([.0-9+-]+)_m_(\d+)_k_([+-]?[0-9.-]+)_nr_([+-]?\d+)"
    match = re.search(pattern1, filename)
    
    if match:
        bv_freq = float(match.group(1))
        omega = float(match.group(2))
        m = int(match.group(3))
        k = float(match.group(4))
        nrchop = int(match.group(5))
    else:
        print(f"Couldn't parse parameters from {filename}")
        return None
    
    # Parse file content
    eigenvalues = []
    is_resolved = []
    indices = []
    
    with open(filename, 'r') as file:
        for line in file:
            # Match the eigenvalue format
            match = re.search(r'II\s*=\s*(\d+)\s*:\s*\(([^,]+),([^)]+)\)\s*-\s*([TF])', line)
            if match:
                index = int(match.group(1))
                real_part = float(match.group(2))
                imag_part = float(match.group(3))
                resolved = match.group(4) == 'T'
                
                indices.append(index)
                eigenvalues.append(complex(real_part, imag_part))
                is_resolved.append(resolved)
    
    return {
        'bv_freq': bv_freq,
        'omega': omega,
        'm': m,
        'k': k,
        'nrchop': nrchop,
        'indices': np.array(indices),
        'eigenvalues': np.array(eigenvalues),
        'is_resolved': np.array(is_resolved)
    }

In [ ]:
# data = parse_eigenvalue_file('../../data/bsnsq_eig_bv_.07_w_.00_m_1_k_+1.0000_nr_+244.output')
# data['eigenvalues']

In [ ]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt

def plot_eigenvalue_spectrum(filename):
    """
    Parses and plots the eigenvalue spectrum from an MLegS output file.
    """
    # Parse the data using your function
    data = parse_eigenvalue_file(filename)
    
    if data is None:
        print(f"Skipping {filename}: Could not parse.")
        return
    
    eigvals = data['eigenvalues']
    resolved = data['is_resolved']
    
    # Split the complex eigenvalues into real and imaginary parts
    res_real = np.real(eigvals[resolved])
    res_imag = np.imag(eigvals[resolved])
    
    unres_real = np.real(eigvals[~resolved])
    unres_imag = np.imag(eigvals[~resolved])
    
    # Create the plot
    fig, ax = plt.subplots(figsize=(10, 7))
    
    # Plot unresolved modes first (in the background)
    ax.scatter(unres_real, unres_imag, color='gray', marker='.', alpha=0.5, 
               label=f'Unresolved ({len(unres_real)})')
    
    # Plot resolved discrete modes on top
    ax.scatter(res_real, res_imag, color='tab:red', marker='.', edgecolors='none', 
               s=40, zorder=3, label=f'Resolved ({len(res_real)})')
    
    # --- NEW LOGIC: Thin crosshair markers and text annotations ---
    
    # Helper to format the complex number string cleanly
    def format_eigval(r, i):
        sign = '+' if i >= 0 else '-'
        return f"{r:.4e} {sign} {abs(i):.4e}i"

    # 1. Most unstable RESOLVED mode
    if len(res_real) > 0:
        max_res_idx = np.argmax(res_real)
        if res_real[max_res_idx] > 0:
            r_val = res_real[max_res_idx]
            i_val = res_imag[max_res_idx]
            val_str = format_eigval(r_val, i_val)
            
            # Large, thin crosshair marker
            ax.scatter(r_val, i_val, color='tab:red', marker='+', 
                       s=40, linewidths=1.5, zorder=5, 
                       label=f'Max Resolved\n({val_str})')
            
            # Label the point directly on the plot with a thin arrow
            ax.annotate(val_str, xy=(r_val, i_val), 
                        xytext=(20, 20), textcoords='offset points',
                        color='tab:blue', fontsize=9, fontweight='bold',
                        bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="tab:blue", alpha=0.8),
                        arrowprops=dict(arrowstyle="->", color='tab:blue', lw=1.0, connectionstyle="arc3,rad=0.2"))

    # 2. Most unstable UNRESOLVED mode
    if len(unres_real) > 0:
        max_unres_idx = np.argmax(unres_real)
        if unres_real[max_unres_idx] > 0:
            r_val = unres_real[max_unres_idx]
            i_val = unres_imag[max_unres_idx]
            val_str = format_eigval(r_val, i_val)
            
            # Large, thin crosshair marker (rotated slightly using 'x' to distinguish from resolved)
            ax.scatter(r_val, i_val, color='tab:gray', marker='x', 
                       s=40, linewidths=1.5, zorder=4, 
                       label=f'Max Unresolved\n({val_str})')
            
            # Label the point directly on the plot with a thin arrow
            ax.annotate(val_str, xy=(r_val, i_val), 
                        xytext=(20, -30), textcoords='offset points',
                        color='tab:orange', fontsize=9, fontweight='bold',
                        bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="tab:orange", alpha=0.8),
                        arrowprops=dict(arrowstyle="->", color='tab:orange', lw=1.0, connectionstyle="arc3,rad=-0.2"))
            
    # -----------------------------------------------------
    
    # Add axes for origin (neutral stability / zero frequency)
    ax.axhline(0, color='black', linewidth=1, linestyle='--', zorder=1)
    ax.axvline(0, color='black', linewidth=1, linestyle='--', zorder=1)
    
    # Formatting and labels
    ax.set_xlabel('Real Part (Growth Rate)')
    ax.set_ylabel('Imaginary Part (Frequency)')
    
    # Dynamic title based on parsed parameters
    title = (f"Eigenvalue Spectrum | m={data['m']}, k={data['k']}, Nr={data['nrchop']}\n"
             f"N={data['bv_freq']}, Omega={data['omega']}")
    ax.set_title(title, pad=15)
    
    ax.grid(True, linestyle=':', alpha=0.6)
    
    # Move legend slightly out of the way if it's getting too big
    ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1), borderaxespad=0.)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# --- Execution Block ---
# Find all matching eigenvalue files in the current directory and plot the first one
# (You can loop through all of them by replacing [0] or pass a specific filename string)
# eigenvalue_files = glob.glob('../../data/bsnsq_eig_bv_.07_w_.00_m_1_k_+1.0000_nr_+244.output')
eigenvalue_files = glob.glob('../../data/bsnsq_eig_bv_5.00_w_.00_m_1_k_+10.0000_nr_+244.output')

if eigenvalue_files:
    print(f"Found {len(eigenvalue_files)} files. Plotting the first one...")
    plot_eigenvalue_spectrum(eigenvalue_files[0])
else:
    print("No eigenvalue files found in the current directory.")

# Spectrum

In [ ]:
import os
import re
import glob
import numpy as np
import matplotlib.pyplot as plt

def parse_eigenvalue_file(filename):
    """
    Parse eigenvalue files from MLegS output, handling the F12.4 Fortran format.
    """
    pattern = r"bsnsq_eig_bv_([+-]?\d+\.\d+)_w_([+-]?\d+\.\d+)_m_([+-]?\d+)_k_([+-]?\d+\.\d+)_nr_([+-]?\d+)\.(txt|output)"
    match = re.search(pattern, filename)
    
    if match:
        bv_freq = float(match.group(1))
        omega = float(match.group(2))
        m = int(match.group(3))
        k = float(match.group(4))
        nrchop = int(match.group(5))
    else:
        print(f"Regex failed to parse: {os.path.basename(filename)}")
        return None
    
    eigenvalues = []
    is_resolved = []
    
    with open(filename, 'r') as file:
        for line in file:
            match = re.search(r'II\s*=\s*(\d+)\s*:\s*\(([^,]+),([^)]+)\)\s*-\s*([TF])', line)
            if match:
                real_part = float(match.group(2))
                imag_part = float(match.group(3))
                resolved = match.group(4) == 'T'
                
                eigenvalues.append(complex(real_part, imag_part))
                is_resolved.append(resolved)
    
    return {
        'bv_freq': bv_freq, 'omega': omega, 'm': m, 'k': k, 'nrchop': nrchop,
        'eigenvalues': np.array(eigenvalues), 'is_resolved': np.array(is_resolved)
    }

def plot_dispersion_cloud(directory='../../data', bv_target=None, m_target=None):
    """
    Plots the dispersion relation, distinctly highlighting Critical Layer Instabilities
    (unresolved but unstable modes) from the continuous spectrum and resolved discrete modes.
    """
    print(f"Searching in absolute directory: {os.path.abspath(directory)}")
    
    if bv_target is not None and m_target is not None:
        file_prefix = f"bsnsq_eig_bv_{bv_target:.4f}_w_*_m_{m_target}_k_*.txt"
        search_pattern = os.path.join(directory, file_prefix)
    else:
        search_pattern = os.path.join(directory, 'bsnsq_eig_bv_*.txt')
        
    files = glob.glob(search_pattern)
    if not files:
        files = glob.glob(search_pattern.replace('.txt', '.output'))
        
    if not files:
        print("ERROR: No eigenvalue files found. Check your directory path!")
        return

    # Aggregate data
    aggregated_data = []
    for f in files:
        data = parse_eigenvalue_file(f)
        if data is None: continue
        if bv_target is not None and not np.isclose(data['bv_freq'], bv_target, atol=1e-3): continue
        if m_target is not None and data['m'] != m_target: continue
        aggregated_data.append(data)
        
    if not aggregated_data:
        print("Files found, but none matched the exact bv_target or m_target.")
        return

    # Sort by k-value
    aggregated_data.sort(key=lambda x: x['k'])
    
    # ---------------------------------------------------------
    # Categorize the spectrum based on physics and resolution
    # ---------------------------------------------------------
    res_k, res_re, res_im = [], [], []         # Smooth, resolved modes
    cl_k, cl_re, cl_im = [], [], []            # Critical Layer (Unresolved & Unstable)
    noise_k, noise_re, noise_im = [], [], []   # Continuous Spectrum (Unresolved & Neutral)
    
    # Small threshold to separate true instability from neutral continuous bands
    growth_threshold = 1e-6 
    
    for d in aggregated_data:
        # Scale the horizontal axis: k / N
        k_scaled = d['k'] / d['bv_freq']
        
        eig = d['eigenvalues']
        res = d['is_resolved']
        
        for i, e in enumerate(eig):
            re_val, im_val = np.real(e), np.imag(e)
            
            # Since the inviscid problem is symmetric, ignore the stable counterparts (re_val < 0)
            if re_val < -1e-5:
                continue
                
            if res[i]:
                res_k.append(k_scaled); res_re.append(re_val); res_im.append(im_val)
            elif re_val > growth_threshold:
                # Physically important Radiative/Critical Layer instability
                cl_k.append(k_scaled); cl_re.append(re_val); cl_im.append(im_val)
            else:
                # Neutral continuous spectrum or non-growing numerical noise
                noise_k.append(k_scaled); noise_re.append(re_val); noise_im.append(im_val)

    # Setup the two-panel figure
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10), sharex=True)
    
    # --- Panel 1: Growth Rate (Real Part) vs K/N ---
    # 1. Background neutral noise (small, faint markers)
    ax1.scatter(noise_k, noise_re, c='gray', s=1, alpha=0.3, marker='o', edgecolors='none', label='Continuous Spectrum (Neutral)')
    # 2. Critical Layer Instabilities (small solid markers to form lines)
    ax1.scatter(cl_k, cl_re, c='tab:red', s=1, alpha=1.0, marker='o', edgecolors='none', label='Critical Layer Instabilities')
    # 3. Resolved discrete modes (small solid markers to form lines)
    ax1.scatter(res_k, res_re, c='tab:blue', s=1, alpha=1.0, marker='o', edgecolors='none', label='Resolved Discrete Modes')
    
    ax1.axhline(0, color='black', linewidth=1, linestyle='--')
    ax1.set_xlabel(r'Scaled Axial Wavenumber ($k/N$)')
    ax1.set_ylabel(r'Growth Rate ($\sigma_r$)')
    ax1.set_title('Instability Growth Rates')
    ax1.grid(True, linestyle=':', alpha=0.6)
    
    # Legend formatting
    leg = ax1.legend(loc='best')
    for handle in leg.legend_handles:
        handle.set_sizes([30.0]) # Make legend markers big enough to read
    
    # Zoom in tightly on the non-negative region
    max_growth = max(res_re + cl_re) if (res_re or cl_re) else 0.1
    ax1.set_ylim(0, max_growth * 1.05)
    ax1.set_xlim(0, max(noise_k + res_k + cl_k))

    # --- Panel 2: Frequency (Imaginary Part) vs K/N ---
    ax2.scatter(noise_k, noise_im, c='gray', s=1, alpha=0.3, marker='o', edgecolors='none')
    ax2.scatter(cl_k, cl_im, c='tab:red', s=1, alpha=1.0, marker='o', edgecolors='none')
    ax2.scatter(res_k, res_im, c='tab:blue', s=1, alpha=1.0, marker='o', edgecolors='none')
    
    ax2.axhline(0, color='black', linewidth=1, linestyle='--')
    ax2.set_xlabel(r'Scaled Axial Wavenumber ($k/N$)')
    ax2.set_ylabel(r'Wave Frequency ($\sigma_i$)')
    ax2.set_title('Dispersion Relation (Frequencies)')
    ax2.grid(True, linestyle=':', alpha=0.6)
    ax2.set_xlim(0, max(noise_k + res_k + cl_k))

    # Global title
    bv_str = aggregated_data[0]['bv_freq']
    m_str = aggregated_data[0]['m']
    fig.suptitle(f'Radiative Instability Spectrum | $N$ = {bv_str}, $m$ = {m_str}', fontsize=14, fontweight='bold', y=1.02)
    
    plt.tight_layout()
    plt.show()

# --- Execution Block ---
plot_dispersion_cloud(directory='../../data/l_10.0_n_1.0_nr_128', bv_target=1.0, m_target=1)

In [ ]:
import os
import re
import glob
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def plot_static_mode_profile(k_over_N, ind, bv_target=5.0, m_target=1, directory='../../data'):
    """
    Finds and plots the spatial velocity and density/buoyancy profiles 
    for a specific mode index (ind) at a specific scaled wavenumber (k/N).
    """
    # 1. Calculate the raw k value expected in the filename
    k_target = k_over_N * bv_target
    
    print(f"Searching for Velocity Data...")
    print(f" -> Target N = {bv_target}, m = {m_target}, k = {k_target} (k/N = {k_over_N})")
    print(f" -> Target Mode IND = {ind}")
    
    # 2. Find the correct velocity subdirectory
    vel_base_dir = os.path.join(directory, 'bsnsq_vel')
    
    # Using glob to find folders matching the known parameters
    dir_pattern = os.path.join(vel_base_dir, f"bv_{bv_target:.4f}_w_*_m_{m_target}_k_*_nr_*")
    candidate_dirs = glob.glob(dir_pattern)
    
    target_dir = None
    for d in candidate_dirs:
        # Extract the exact 'k' value from the folder name to avoid floating point mismatch
        match = re.search(r'_k_([+-]?\d+\.\d+)_nr_', d)
        if match:
            k_val = float(match.group(1))
            if np.isclose(k_val, k_target, atol=1e-4):
                target_dir = d
                break
                
    if not target_dir:
        print("ERROR: Could not find a matching velocity directory.")
        print(f"Looked in: {vel_base_dir}")
        return
        
    # 3. Target the exact text file
    file_path = os.path.join(target_dir, f"ind_{ind}.txt")
    
    if not os.path.exists(file_path):
        print(f"ERROR: Data file not found for IND={ind}!")
        print(f"Expected path: {file_path}")
        return
        
    print(f"Loading data from: {file_path}")
    
    # 4. Load the data
    try:
        data = np.loadtxt(file_path, delimiter=',')
    except Exception as e:
        print(f"Error reading file: {e}")
        return
        
    r = data[:, 0]
    ur_r, ur_i = data[:, 1], data[:, 2]
    up_r, up_i = data[:, 3], data[:, 4]
    uz_r, uz_i = data[:, 5], data[:, 6]
    has_b = data.shape[1] >= 9
    
    if has_b:
        b_r, b_i = data[:, 7], data[:, 8]
    else:
        b_r, b_i = [], []

    # 5. Plotly Figure Setup
    fig = make_subplots(
        rows=1, cols=4,
        subplot_titles=("u_r", "u_\u03B8", "u_z", "b (Density)"),
        horizontal_spacing=0.04
    )

    line_r = dict(color='black', width=2)
    line_i = dict(color='red', width=2, dash='dash')

    # Trace 1: ur
    fig.add_trace(go.Scatter(x=r, y=ur_r, mode='lines', line=line_r, name='Real'), row=1, col=1)
    fig.add_trace(go.Scatter(x=r, y=ur_i, mode='lines', line=line_i, name='Imag'), row=1, col=1)

    # Trace 2: up
    fig.add_trace(go.Scatter(x=r, y=up_r, mode='lines', line=line_r, showlegend=False), row=1, col=2)
    fig.add_trace(go.Scatter(x=r, y=up_i, mode='lines', line=line_i, showlegend=False), row=1, col=2)

    # Trace 3: uz
    fig.add_trace(go.Scatter(x=r, y=uz_r, mode='lines', line=line_r, showlegend=False), row=1, col=3)
    fig.add_trace(go.Scatter(x=r, y=uz_i, mode='lines', line=line_i, showlegend=False), row=1, col=3)

    # Trace 4: b
    if has_b:
        fig.add_trace(go.Scatter(x=r, y=b_r, mode='lines', line=line_r, showlegend=False), row=1, col=4)
        fig.add_trace(go.Scatter(x=r, y=b_i, mode='lines', line=line_i, showlegend=False), row=1, col=4)

    # 6. Layout adjustments
    fig.update_layout(
        height=450, 
        width=1600,
        title_text=f'Spatial Profiles | N = {bv_target}, m = {m_target} | k/N = {k_over_N} | Mode IND = {ind}',
        title_font=dict(size=18, family="Arial", color="black"),
        template='plotly_white',
        margin=dict(t=70, b=40, l=40, r=40)
    )
    
    # Clean axes and set radius range to [0, 12]
    for i in range(1, 5):
        fig.update_xaxes(title_text="Radius (r)", range=[0, 12], row=1, col=i)
        fig.update_yaxes(zeroline=True, zerolinecolor='lightgray', zerolinewidth=1, row=1, col=i)

    fig.show()


In [ ]:
plot_static_mode_profile(k_over_N=2.00, ind=46, bv_target=1.0, m_target=1, directory='../../data/l_10.0_n_1.0_nr_128')

In [ ]:
import os
import re
import glob
import numpy as np
import matplotlib.pyplot as plt

def parse_eigenvalue_file(filename):
    pattern = r"bsnsq_eig_bv_([+-]?\d+\.\d+)_w_([+-]?\d+\.\d+)_m_([+-]?\d+)_k_([+-]?\d+\.\d+)_nr_([+-]?\d+)\.(txt|output)"
    match = re.search(pattern, filename)
    if match:
        return {
            'bv_freq': float(match.group(1)), 'omega': float(match.group(2)),
            'm': int(match.group(3)), 'k': float(match.group(4)), 'nrchop': int(match.group(5)),
            'eigenvalues': [], 'is_resolved': []
        }
    return None

def get_data(filename):
    data = parse_eigenvalue_file(filename)
    if not data: return None
    with open(filename, 'r') as file:
        for line in file:
            match = re.search(r'II\s*=\s*(\d+)\s*:\s*\(([^,]+),([^)]+)\)\s*-\s*([TF])', line)
            if match:
                data['eigenvalues'].append(complex(float(match.group(2)), float(match.group(3))))
                data['is_resolved'].append(match.group(4) == 'T')
    data['eigenvalues'] = np.array(data['eigenvalues'])
    data['is_resolved'] = np.array(data['is_resolved'])
    return data

def Omega(r):
    """Normalized Lamb-Oseen Vortex Angular Velocity"""
    with np.errstate(divide='ignore', invalid='ignore'):
        val = (1 - np.exp(-r**2)) / r**2
    return np.where(r == 0, 1.0, val)

def plot_dispersion_cloud(directory='../../data', bv_target=5.0, m_target=1):
    
    search_pattern = os.path.join(directory, f"bsnsq_eig_bv_{bv_target:.4f}_w_*_m_{m_target}_k_*.txt")
    files = glob.glob(search_pattern)
    if not files: files = glob.glob(search_pattern.replace('.txt', '.output'))
    if not files:
        print("ERROR: No eigenvalue files found.")
        return

    aggregated_data = [d for f in files if (d := get_data(f)) and np.isclose(d['bv_freq'], bv_target) and d['m'] == m_target]
    if not aggregated_data: return
    aggregated_data.sort(key=lambda x: x['k'])

    # Arrays for categories
    res_k, res_re, res_im = [], [], []             # Blue (Resolved)
    cl_unst_k, cl_unst_re, cl_unst_im = [], [], [] # Red (CL Unstable)
    cl_dec_k, cl_dec_re, cl_dec_im = [], [], []    # Green (CL Decaying)
    s0_unst_k, s0_unst_re, s0_unst_im = [], [], [] # Brown (S=0 Unstable)
    s0_dec_k, s0_dec_re, s0_dec_im = [], [], []    # Olive (S=0 Decaying)
    noise_k, noise_re, noise_im = [], [], []       # Gray (Neutral Noise)

    growth_threshold = 1e-13
    r_grid = np.linspace(0, 50, 2000) # High-res grid to find S=0 global bounds
    
    # Track the theoretical bounds for plotting the dashed lines
    k_line_scaled = []
    upper_max_line, upper_min_line = [], []
    lower_max_line, lower_min_line = [], []

    for d in aggregated_data:
        k_val = d['k']
        k_scaled = k_val / d['bv_freq']
        k_line_scaled.append(k_scaled)
        
        # Calculate Exact S=0 Continuous Spectrum Envelopes for this k
        f_plus = -m_target * Omega(r_grid) + (m_target * d['bv_freq']) / np.sqrt(k_val**2 * r_grid**2 + m_target**2)
        f_minus = -m_target * Omega(r_grid) - (m_target * d['bv_freq']) / np.sqrt(k_val**2 * r_grid**2 + m_target**2)
        
        up_max, up_min = np.max(f_plus), np.min(f_plus)
        low_max, low_min = np.max(f_minus), np.min(f_minus)
        
        upper_max_line.append(up_max); upper_min_line.append(up_min)
        lower_max_line.append(low_max); lower_min_line.append(low_min)

        for i, e in enumerate(d['eigenvalues']):
            re_val, im_val = np.real(e), np.imag(e)
            is_non_neutral = abs(re_val) > growth_threshold
            
            # Physics Checks
            in_CL = (-m_target - 1e-3 <= im_val <= 1e-3)
            in_S0 = (up_min - 1e-3 <= im_val <= up_max + 1e-3) or (low_min - 1e-3 <= im_val <= low_max + 1e-3)
            
            if d['is_resolved'][i]:
                res_k.append(k_scaled); res_re.append(re_val); res_im.append(im_val)
            elif is_non_neutral and in_CL:
                if re_val > 0:
                    cl_unst_k.append(k_scaled); cl_unst_re.append(re_val); cl_unst_im.append(im_val)
                else:
                    cl_dec_k.append(k_scaled); cl_dec_re.append(re_val); cl_dec_im.append(im_val)
            elif is_non_neutral and in_S0:
                if re_val > 0:
                    s0_unst_k.append(k_scaled); s0_unst_re.append(re_val); s0_unst_im.append(im_val)
                else:
                    s0_dec_k.append(k_scaled); s0_dec_re.append(re_val); s0_dec_im.append(im_val)
            else:
                noise_k.append(k_scaled); noise_re.append(re_val); noise_im.append(im_val)

    # Setup the two-panel figure
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10), sharex=True)
    
    # --- Panel 1: Growth Rate ---
    ax1.scatter(noise_k, noise_re, c='gray', s=1, alpha=0.3, marker='o', edgecolors='none')
    
    ax1.scatter(s0_dec_k, s0_dec_re, c='olive', s=2, alpha=1.0, marker='o', edgecolors='none', label='S=0 Singular Modes (Decaying)')
    ax1.scatter(s0_unst_k, s0_unst_re, c='saddlebrown', s=2, alpha=1.0, marker='o', edgecolors='none', label='S=0 Singular Modes (Unstable)')
    
    ax1.scatter(cl_dec_k, cl_dec_re, c='tab:green', s=2, alpha=1.0, marker='o', edgecolors='none', label='Barotropic CL Modes (Decaying)')
    ax1.scatter(cl_unst_k, cl_unst_re, c='tab:red', s=2, alpha=1.0, marker='o', edgecolors='none', label='Barotropic CL Modes (Unstable)')
    
    ax1.scatter(res_k, res_re, c='tab:blue', s=2, alpha=1.0, marker='o', edgecolors='none', label='Resolved Discrete Modes')
    
    ax1.axhline(0, color='black', linewidth=1, linestyle='--')
    ax1.set_xlabel(r'Scaled Axial Wavenumber ($k/N$)')
    ax1.set_ylabel(r'Growth Rate ($\sigma_r$)')
    ax1.set_title('Instability Growth Rates')
    ax1.grid(True, linestyle=':', alpha=0.6)
    
    leg = ax1.legend(loc='best')
    for handle in leg.legend_handles: handle.set_sizes([30.0])
    
    max_growth = max([abs(x) for x in res_re + cl_unst_re + cl_dec_re + s0_unst_re + s0_dec_re] + [0.01])
    ax1.set_ylim(-max_growth * 1.05, max_growth * 1.05)
    ax1.set_xlim(0, max(k_line_scaled))

    # --- Panel 2: Frequency ---
    ax2.scatter(noise_k, noise_im, c='gray', s=1, alpha=0.3, marker='o', edgecolors='none')
    
    ax2.scatter(s0_dec_k, s0_dec_im, c='olive', s=2, alpha=1.0, marker='o', edgecolors='none')
    ax2.scatter(s0_unst_k, s0_unst_im, c='saddlebrown', s=2, alpha=1.0, marker='o', edgecolors='none')
    
    ax2.scatter(cl_dec_k, cl_dec_im, c='tab:green', s=2, alpha=1.0, marker='o', edgecolors='none')
    ax2.scatter(cl_unst_k, cl_unst_im, c='tab:red', s=2, alpha=1.0, marker='o', edgecolors='none')
    
    ax2.scatter(res_k, res_im, c='tab:blue', s=2, alpha=1.0, marker='o', edgecolors='none')
    
    # Plotting the Analytical Bounds
    ax2.plot(k_line_scaled, upper_max_line, 'k--', linewidth=1.5, label='Theoretical S=0 Bounds')
    ax2.plot(k_line_scaled, upper_min_line, 'k--', linewidth=1.5)
    ax2.plot(k_line_scaled, lower_max_line, 'k--', linewidth=1.5)
    ax2.plot(k_line_scaled, lower_min_line, 'k--', linewidth=1.5)
    
    ax2.axhline(0, color='red', linestyle='--', linewidth=1.5, label='Barotropic CL Bounds')
    ax2.axhline(-m_target, color='red', linestyle='--', linewidth=1.5)

    ax2.axhline(0, color='black', linewidth=1, linestyle='-')
    ax2.set_xlabel(r'Scaled Axial Wavenumber ($k/N$)')
    ax2.set_ylabel(r'Wave Frequency ($\sigma_i$)')
    ax2.set_title('Dispersion Relation (Frequencies)')
    ax2.grid(True, linestyle=':', alpha=0.6)
    ax2.legend(loc='best')

    fig.suptitle(f'Radiative Instability Spectrum | $N$ = {bv_target}, $m$ = {m_target}', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

# --- Execution Block ---
plot_dispersion_cloud(directory='../../data/l_10.0_n_1.0_nr_128', bv_target=1.0, m_target=1)

# Lamb-Oseen vortex

In [ ]:
import os
import re
import glob
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

def base_flow_profiles(r):
    """Calculates the Lamb-Oseen vortex base flow properties."""
    with np.errstate(divide='ignore', invalid='ignore'):
        omega = (1 - np.exp(-r**2)) / r**2
        zeta = 2 * np.exp(-r**2)
        
    # Handle origin (r=0) safely
    omega = np.where(r == 0, 1.0, omega)
    zeta = np.where(r == 0, 2.0, zeta)
    delta = 2 * omega * zeta
    return omega, delta

def plot_static_mode_profile(k_over_N, ind, bv_target=5.0, m_target=1, directory='../../data', r_max=12.0):
    """
    Plots the spatial velocity profiles and a WKBJ/Critical Layer diagnostic 
    frequency map, including an automated zoom-in of the 3-root region.
    """
    k_target = k_over_N * bv_target
    
    print(f"Searching for Data...")
    print(f" -> Target N = {bv_target}, m = {m_target}, k = {k_target} (k/N = {k_over_N})")
    print(f" -> Target Mode IND = {ind}")
    
    # --- 1. FIND VELOCITY DIRECTORY ---
    vel_base_dir = os.path.join(directory, 'bsnsq_vel')
    dir_pattern = os.path.join(vel_base_dir, f"bv_{bv_target:.4f}_w_*_m_{m_target}_k_*_nr_*")
    candidate_dirs = glob.glob(dir_pattern)
    
    target_dir = None
    for d in candidate_dirs:
        match = re.search(r'_k_([+-]?\d+\.\d+)_nr_', d)
        if match and np.isclose(float(match.group(1)), k_target, atol=1e-4):
            target_dir = d
            break
            
    if not target_dir:
        print("ERROR: Could not find a matching velocity directory.")
        return
        
    # --- 2. LOAD VELOCITY DATA ---
    file_path = os.path.join(target_dir, f"ind_{ind}.txt")
    if not os.path.exists(file_path):
        print(f"ERROR: Velocity data file not found for IND={ind}!")
        return
        
    try:
        data = np.loadtxt(file_path, delimiter=',')
    except Exception as e:
        print(f"Error reading velocity file: {e}")
        return
        
    r = data[:, 0]
    ur_r, ur_i = data[:, 1], data[:, 2]
    up_r, up_i = data[:, 3], data[:, 4]
    uz_r, uz_i = data[:, 5], data[:, 6]
    has_b = data.shape[1] >= 9
    if has_b:
        b_r, b_i = data[:, 7], data[:, 8]
    else:
        b_r, b_i = [], []

    # --- 3. EXTRACT EXACT EIGENVALUE FREQUENCY ---
    folder_name = os.path.basename(target_dir) 
    eig_filename = f"bsnsq_eig_{folder_name}.txt"
    eig_file_path = os.path.join(directory, eig_filename)
    if not os.path.exists(eig_file_path):
        eig_file_path = eig_file_path.replace('.txt', '.output')
        
    mode_freq = None
    if os.path.exists(eig_file_path):
        with open(eig_file_path, 'r') as f:
            for line in f:
                match = re.search(rf'II\s*=\s*{ind}\s*:\s*\(([^,]+),([^)]+)\)', line)
                if match:
                    mode_freq = float(match.group(2))
                    break

    # --- 4. CALCULATE BASE FLOW FREQUENCY CURVES ---
    r_dense = np.linspace(0, r_max, 2000)
    omega_base, delta_base = base_flow_profiles(r_dense)
    
    tp_plus = -m_target * omega_base + np.sqrt(delta_base)
    tp_minus = -m_target * omega_base - np.sqrt(delta_base)
    barotropic_cl = -m_target * omega_base
    baroclinic_plus = -m_target * omega_base + bv_target
    baroclinic_minus = -m_target * omega_base - bv_target
    
    s_term = (m_target * bv_target) / np.sqrt(k_target**2 * r_dense**2 + m_target**2)
    s_plus = -m_target * omega_base + s_term
    s_minus = -m_target * omega_base - s_term

    # --- 5. FIGURE SETUP ---
    fig = plt.figure(figsize=(16, 12))
    gs = GridSpec(4, 2, width_ratios=[1.2, 1], hspace=0.4, wspace=0.25)
    
    # Left Panel: Frequency Map
    ax_freq = fig.add_subplot(gs[:, 0])
    
    ax_freq.plot(r_dense, tp_plus, color='purple', linestyle='--', linewidth=2, label='WKBJ Turning Pt (+)')
    ax_freq.plot(r_dense, tp_minus, color='purple', linestyle='--', linewidth=2, label='WKBJ Turning Pt (-)')
    ax_freq.plot(r_dense, barotropic_cl, color='red', linestyle='-', linewidth=2, label=r'Barotropic CL ($\Phi=0$)')
    ax_freq.plot(r_dense, baroclinic_plus, color='green', linestyle=':', linewidth=2, label=r'Baroclinic CL ($+N$)')
    ax_freq.plot(r_dense, baroclinic_minus, color='green', linestyle=':', linewidth=2, label=r'Baroclinic CL ($-N$)')
    ax_freq.plot(r_dense, s_plus, color='saddlebrown', linestyle='-.', linewidth=2, label=r'Regular Singularity $S(r)=0$ (+)')
    ax_freq.plot(r_dense, s_minus, color='saddlebrown', linestyle='-.', linewidth=2, label=r'Regular Singularity $S(r)=0$ (-)')

    if mode_freq is not None:
        ax_freq.axhline(y=mode_freq, color='black', linewidth=2, linestyle='-', label=f'Mode Freq: {mode_freq:.4f}')

    ax_freq.set_xlim(0, r_max)
    ax_freq.set_xlabel('Radius (r)', fontsize=12)
    ax_freq.set_ylabel(r'Wave Frequency ($\sigma_i$)', fontsize=12)
    ax_freq.set_title(f'Diagnostic Frequency Map | k/N = {k_over_N}', fontsize=14)
    ax_freq.grid(True, linestyle=':', alpha=0.6)
    ax_freq.legend(loc='lower right', fontsize=10)

    # --- 6. ADD ZOOM-IN INSET FOR 3-ROOT REGION ---
    # Dynamically find the shallow potential well bounding the 3-root region
    x_zmin, x_zmax = 1.0, r_max
    z_mask = (r_dense >= x_zmin) & (r_dense <= x_zmax)
    
    # The 3-root region exists where tp_plus dips to its local minimum before asymptoting
    y_zmin = np.min(tp_plus[z_mask]) - 0.02
    y_zmax = 0.05 
    
    # Create inset axes [x0, y0, width, height] in normalized axes coordinates
    # Changed to [0.55, 0.55, 0.42, 0.42] to anchor it in the upper right
    axins = ax_freq.inset_axes([0.55, 0.65, 0.42, 0.32])
    
    # Re-plot everything on the inset
    axins.plot(r_dense, tp_plus, color='purple', linestyle='--', linewidth=2)
    axins.plot(r_dense, tp_minus, color='purple', linestyle='--', linewidth=2)
    axins.plot(r_dense, barotropic_cl, color='red', linestyle='-', linewidth=2)
    axins.plot(r_dense, baroclinic_plus, color='green', linestyle=':', linewidth=2)
    axins.plot(r_dense, baroclinic_minus, color='green', linestyle=':', linewidth=2)
    axins.plot(r_dense, s_plus, color='saddlebrown', linestyle='-.', linewidth=2)
    axins.plot(r_dense, s_minus, color='saddlebrown', linestyle='-.', linewidth=2)
    if mode_freq is not None:
        axins.axhline(y=mode_freq, color='black', linewidth=2, linestyle='-')

    axins.set_xlim(x_zmin, x_zmax)
    axins.set_ylim(y_zmin, y_zmax)
    axins.grid(True, linestyle=':', alpha=0.6)
    axins.set_title("Zoom: 3-Root Potential Well", fontsize=10)
    
    # Draw the connector lines
    ax_freq.indicate_inset_zoom(axins, edgecolor="black")

    # --- 7. RIGHT PANELS: VELOCITY PROFILES ---
    axes_vel = [fig.add_subplot(gs[i, 1]) for i in range(4)]
    titles = [r'$u_r$', r'$u_\theta$', r'$u_z$', r'$b$ (Density)']
    
    y_data_pairs = [(ur_r, ur_i), (up_r, up_i), (uz_r, uz_i), (b_r, b_i) if has_b else ([], [])]
    
    for i, ax in enumerate(axes_vel):
        real_data, imag_data = y_data_pairs[i]
        
        if len(real_data) > 0:
            ax.plot(r, real_data, color='black', linewidth=2, label='Real' if i==0 else "")
            ax.plot(r, imag_data, color='blue', linewidth=2, linestyle='--', label='Imag' if i==0 else "")
            
        ax.set_xlim(0, r_max)
        ax.axhline(0, color='gray', linewidth=1)
        ax.set_ylabel(titles[i], fontsize=12)
        ax.grid(True, linestyle=':', alpha=0.5)
        
        if i == 3:
            ax.set_xlabel('Radius (r)', fontsize=12)
        else:
            ax.tick_params(labelbottom=False) 
        
        if i == 0:
            ax.set_title(f'Spatial Eigenfunctions', fontsize=14)
            ax.legend(loc='upper right')

    fig.suptitle(f'Mode Structure | N = {bv_target}, m = {m_target}, k/N = {k_over_N} | IND = {ind}', fontsize=16, fontweight='bold', y=0.95)
    
    plt.show()

In [ ]:
plot_static_mode_profile(k_over_N=0.05, ind=277, bv_target=1.0, m_target=1, directory='../../data/l_10.0_n_1.0_nr_128', r_max=4.0)

In [ ]:
import os
import re
import glob
import numpy as np
import matplotlib.pyplot as plt

def parse_eigenvalue_file(filename):
    pattern = r"bsnsq_eig_bv_([+-]?\d+\.\d+)_w_([+-]?\d+\.\d+)_m_([+-]?\d+)_k_([+-]?\d+\.\d+)_nr_([+-]?\d+)\.(txt|output)"
    match = re.search(pattern, filename)
    if match:
        return {
            'bv_freq': float(match.group(1)), 'omega': float(match.group(2)),
            'm': int(match.group(3)), 'k': float(match.group(4)), 'nrchop': int(match.group(5)),
            'eigenvalues': [], 'is_resolved': []
        }
    return None

def get_data(filename):
    data = parse_eigenvalue_file(filename)
    if not data: return None
    with open(filename, 'r') as file:
        for line in file:
            match = re.search(r'II\s*=\s*(\d+)\s*:\s*\(([^,]+),([^)]+)\)\s*-\s*([TF])', line)
            if match:
                data['eigenvalues'].append(complex(float(match.group(2)), float(match.group(3))))
                data['is_resolved'].append(match.group(4) == 'T')
    data['eigenvalues'] = np.array(data['eigenvalues'])
    data['is_resolved'] = np.array(data['is_resolved'])
    return data

def plot_dispersion_cloud(directory='../../data', bv_target=5.0, m_target=1):
    
    search_pattern = os.path.join(directory, f"bsnsq_eig_bv_{bv_target:.4f}_w_*_m_{m_target}_k_*.txt")
    files = glob.glob(search_pattern)
    if not files: files = glob.glob(search_pattern.replace('.txt', '.output'))
    if not files:
        print("ERROR: No eigenvalue files found.")
        return

    aggregated_data = [d for f in files if (d := get_data(f)) and np.isclose(d['bv_freq'], bv_target) and d['m'] == m_target]
    if not aggregated_data: return
    aggregated_data.sort(key=lambda x: x['k'])

    # Arrays for the two categories we care about
    res_k, res_re, res_im = [], [], []   # Blue (Resolved Discrete Modes)
    cl_k, cl_re, cl_im = [], [], []      # Red (Unstable Barotropic CL Modes)

    growth_threshold = 1e-13

    for d in aggregated_data:
        k_scaled = d['k'] / d['bv_freq']
        
        for i, e in enumerate(d['eigenvalues']):
            re_val, im_val = np.real(e), np.imag(e)
            
            # Skip all strictly stable (decaying) modes
            if re_val < -1e-5:
                continue
                
            # Check if frequency is within the Barotropic CL band [0, -m]
            in_CL = (-m_target - 1e-3 <= im_val <= 1e-3)
            
            if d['is_resolved'][i]:
                res_k.append(k_scaled); res_re.append(re_val); res_im.append(im_val)
            elif re_val > growth_threshold and in_CL:
                # Only keep unresolved modes if they are unstable AND physically valid CL modes
                cl_k.append(k_scaled); cl_re.append(re_val); cl_im.append(im_val)

    # Setup the two-panel figure
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10), sharex=True)
    
    # --- Panel 1: Growth Rate ---
    ax1.scatter(cl_k, cl_re, c='tab:red', s=4, alpha=1.0, marker='o', edgecolors='none', label='Barotropic CL Modes (Unstable)')
    ax1.scatter(res_k, res_re, c='tab:blue', s=4, alpha=1.0, marker='o', edgecolors='none', label='Resolved Discrete Modes')
    
    ax1.axhline(0, color='black', linewidth=1, linestyle='--')
    ax1.set_xlabel(r'Scaled Axial Wavenumber ($k/N$)')
    ax1.set_ylabel(r'Growth Rate ($\sigma_r$)')
    ax1.set_title('Instability Growth Rates')
    ax1.grid(True, linestyle=':', alpha=0.6)
    
    leg = ax1.legend(loc='best')
    for handle in leg.legend_handles: handle.set_sizes([30.0])
    
    # Lock y-axis to non-negative growth rates
    max_growth = max([abs(x) for x in res_re + cl_re] + [0.01])
    ax1.set_ylim(0, max_growth * 1.05)
    
    if res_k or cl_k:
        ax1.set_xlim(0, max(res_k + cl_k) * 1.02)

    # --- Panel 2: Frequency ---
    ax2.scatter(cl_k, cl_im, c='tab:red', s=4, alpha=1.0, marker='o', edgecolors='none')
    ax2.scatter(res_k, res_im, c='tab:blue', s=4, alpha=1.0, marker='o', edgecolors='none')
    
    # Plotting the Analytical Bounds for Barotropic CL
    ax2.axhline(0, color='red', linestyle='--', linewidth=1.5, label='Barotropic CL Bounds')
    ax2.axhline(-m_target, color='red', linestyle='--', linewidth=1.5)
    ax2.axhline(0, color='black', linewidth=1, linestyle='-')

    ax2.set_xlabel(r'Scaled Axial Wavenumber ($k/N$)')
    ax2.set_ylabel(r'Wave Frequency ($\sigma_i$)')
    ax2.set_title('Dispersion Relation (Frequencies)')
    ax2.grid(True, linestyle=':', alpha=0.6)
    ax2.legend(loc='best')

    fig.suptitle(f'Radiative Instability Spectrum | $N$ = {bv_target}, $m$ = {m_target}', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()


In [ ]:
plot_dispersion_cloud(directory='../../data/l_10.0_n_1.0_nr_128', bv_target=1.0, m_target=1)

In [ ]:
import os
import re
import glob
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def parse_eigenvalue_file(filename):
    pattern = r"bsnsq_eig_bv_([+-]?\d+\.\d+)_w_([+-]?\d+\.\d+)_m_([+-]?\d+)_k_([+-]?\d+\.\d+)_nr_([+-]?\d+)\.(txt|output)"
    match = re.search(pattern, filename)
    if match:
        return {
            'filepath': filename, 'bv_freq': float(match.group(1)), 'omega': float(match.group(2)),
            'm': int(match.group(3)), 'k': float(match.group(4)), 'nrchop': int(match.group(5)),
            'eigenvalues': [], 'is_resolved': [], 'indices': []
        }
    return None

def get_data(filename):
    data = parse_eigenvalue_file(filename)
    if not data: return None
    with open(filename, 'r') as file:
        for line in file:
            match = re.search(r'II\s*=\s*(\d+)\s*:\s*\(([^,]+),([^)]+)\)\s*-\s*([TF])', line)
            if match:
                data['indices'].append(int(match.group(1)))
                data['eigenvalues'].append(complex(float(match.group(2)), float(match.group(3))))
                data['is_resolved'].append(match.group(4) == 'T')
    data['indices'] = np.array(data['indices'])
    data['eigenvalues'] = np.array(data['eigenvalues'])
    data['is_resolved'] = np.array(data['is_resolved'])
    return data

def base_flow_profiles(r):
    with np.errstate(divide='ignore', invalid='ignore'):
        omega = (1 - np.exp(-r**2)) / r**2
        zeta = 2 * np.exp(-r**2)
    omega = np.where(r == 0, 1.0, omega)
    zeta = np.where(r == 0, 2.0, zeta)
    delta = 2 * omega * zeta
    return omega, delta

def create_interactive_full_dashboard(directory='../../data', bv_target=5.0, m_target=1, r_max=12.0):
    
    # --- 1. FILE AGGREGATION & FILTERING ---
    search_pattern = os.path.join(directory, f"bsnsq_eig_bv_{bv_target:.4f}_w_*_m_{m_target}_k_*.txt")
    files = glob.glob(search_pattern)
    if not files: files = glob.glob(search_pattern.replace('.txt', '.output'))
    if not files:
        print("ERROR: No eigenvalue files found.")
        return

    aggregated_data = [d for f in files if (d := get_data(f)) and np.isclose(d['bv_freq'], bv_target) and d['m'] == m_target]
    if not aggregated_data: return
    aggregated_data.sort(key=lambda x: x['k'])

    res_k, res_re, res_im, res_cdata = [], [], [], []
    cl_k, cl_re, cl_im, cl_cdata = [], [], [], []
    growth_threshold = 1e-13
    
    # Set the desired resolution for k/N
    dk_over_N = 0.5

    for d in aggregated_data:
        k_scaled = d['k'] / d['bv_freq']
        
        # Filter to enforce the delta k / N resolution
        # We check the remainder against a small tolerance to handle floating-point math
        remainder = k_scaled % dk_over_N
        if remainder > 1e-3 and remainder < dk_over_N - 1e-3:
            continue
            
        folder_name = os.path.basename(d['filepath']).replace('bsnsq_eig_', '').replace('.txt', '').replace('.output', '')
        
        for i, e in enumerate(d['eigenvalues']):
            re_val, im_val = np.real(e), np.imag(e)
            if re_val < -1e-5: continue # Skip stable
            
            # Customdata structure: (directory, folder_name, mode_idx, k_raw, freq)
            cdata = (directory, folder_name, d['indices'][i], d['k'], im_val)
            in_CL = (-m_target - 1e-3 <= im_val <= 1e-3)
            
            if d['is_resolved'][i]:
                res_k.append(k_scaled); res_re.append(re_val); res_im.append(im_val); res_cdata.append(cdata)
            elif re_val > growth_threshold and in_CL:
                cl_k.append(k_scaled); cl_re.append(re_val); cl_im.append(im_val); cl_cdata.append(cdata)

    # --- 2. BASE FLOW PROFILES (STATIC) ---
    r_dense = np.linspace(0, r_max, 2000)
    omega_base, delta_base = base_flow_profiles(r_dense)
    tp_plus = -m_target * omega_base + np.sqrt(delta_base)
    tp_minus = -m_target * omega_base - np.sqrt(delta_base)
    barotropic_cl = -m_target * omega_base
    baroclinic_plus = -m_target * omega_base + bv_target
    baroclinic_minus = -m_target * omega_base - bv_target
    
    # Calculate bounds for the zoom region
    x_zmin, x_zmax = 1.0, r_max
    z_mask = (r_dense >= x_zmin) & (r_dense <= x_zmax)
    y_zmin = np.min(tp_plus[z_mask]) - 0.02
    y_zmax = 0.05

    # --- 3. PLOTLY FIGURE SETUP ---
    fig = make_subplots(
        rows=3, cols=4,
        row_heights=[0.3, 0.45, 0.25],
        # Split row 2 into two subplots (each taking 2 cols)
        specs=[[{"colspan": 2}, None, {"colspan": 2}, None],
               [{"colspan": 2}, None, {"colspan": 2}, None],
               [{}, {}, {}, {}]],
        subplot_titles=("Growth Rate (\u03C3_r)", "Wave Frequency (\u03C3_i)", 
                        "Diagnostic Frequency Map", "Zoom: 3-Root Potential Well",
                        "u_r", "u_\u03B8", "u_z", "b (Density)"),
        vertical_spacing=0.1, horizontal_spacing=0.04
    )
    
    htemp = "k/N: %{x:.4f}<br>Value: %{y:.4e}<br>Mode ID: %{customdata[2]}<extra></extra>"

    # ROW 1: Dispersion [Traces 0, 1, 2, 3]
    fig.add_trace(go.Scatter(x=cl_k, y=cl_re, mode='markers', marker=dict(color='#d62728', size=2), name='Barotropic CL', customdata=cl_cdata, hovertemplate=htemp), row=1, col=1)
    fig.add_trace(go.Scatter(x=res_k, y=res_re, mode='markers', marker=dict(color='#1f77b4', size=2), name='Resolved', customdata=res_cdata, hovertemplate=htemp), row=1, col=1)

    fig.add_trace(go.Scatter(x=cl_k, y=cl_im, mode='markers', marker=dict(color='#d62728', size=2), showlegend=False, customdata=cl_cdata, hovertemplate=htemp), row=1, col=3)
    fig.add_trace(go.Scatter(x=res_k, y=res_im, mode='markers', marker=dict(color='#1f77b4', size=2), showlegend=False, customdata=res_cdata, hovertemplate=htemp), row=1, col=3)

    # ROW 2 LEFT: Main Frequency Map [Traces 4 to 11]
    dash_p = dict(color='purple', dash='dash', width=2)
    dash_g = dict(color='green', dash='dot', width=2)
    dash_br = dict(color='saddlebrown', dash='dashdot', width=2)
    
    fig.add_trace(go.Scatter(x=r_dense, y=tp_plus, mode='lines', line=dash_p, name='WKBJ Turning Pt (+)'), row=2, col=1)
    fig.add_trace(go.Scatter(x=r_dense, y=tp_minus, mode='lines', line=dash_p, name='WKBJ Turning Pt (-)'), row=2, col=1)
    fig.add_trace(go.Scatter(x=r_dense, y=barotropic_cl, mode='lines', line=dict(color='red', width=2), name='Barotropic CL'), row=2, col=1)
    fig.add_trace(go.Scatter(x=r_dense, y=baroclinic_plus, mode='lines', line=dash_g, name='Baroclinic CL (+N)'), row=2, col=1)
    fig.add_trace(go.Scatter(x=r_dense, y=baroclinic_minus, mode='lines', line=dash_g, name='Baroclinic CL (-N)'), row=2, col=1)
    fig.add_trace(go.Scatter(x=[], y=[], mode='lines', line=dash_br, name='S(r)=0 (+)'), row=2, col=1)
    fig.add_trace(go.Scatter(x=[], y=[], mode='lines', line=dash_br, name='S(r)=0 (-)'), row=2, col=1)
    fig.add_trace(go.Scatter(x=[], y=[], mode='lines', line=dict(color='black', width=2), name='Selected Mode Freq'), row=2, col=1)

    # ROW 2 RIGHT: Zoomed Frequency Map [Traces 12 to 19]
    fig.add_trace(go.Scatter(x=r_dense, y=tp_plus, mode='lines', line=dash_p, showlegend=False), row=2, col=3)
    fig.add_trace(go.Scatter(x=r_dense, y=tp_minus, mode='lines', line=dash_p, showlegend=False), row=2, col=3)
    fig.add_trace(go.Scatter(x=r_dense, y=barotropic_cl, mode='lines', line=dict(color='red', width=2), showlegend=False), row=2, col=3)
    fig.add_trace(go.Scatter(x=r_dense, y=baroclinic_plus, mode='lines', line=dash_g, showlegend=False), row=2, col=3)
    fig.add_trace(go.Scatter(x=r_dense, y=baroclinic_minus, mode='lines', line=dash_g, showlegend=False), row=2, col=3)
    fig.add_trace(go.Scatter(x=[], y=[], mode='lines', line=dash_br, showlegend=False), row=2, col=3)
    fig.add_trace(go.Scatter(x=[], y=[], mode='lines', line=dash_br, showlegend=False), row=2, col=3)
    fig.add_trace(go.Scatter(x=[], y=[], mode='lines', line=dict(color='black', width=2), showlegend=False), row=2, col=3)

    # ROW 3: Velocities [Traces 20 to 27]
    empty_x, empty_y = [], []
    line_r = dict(color='black', width=2); line_i = dict(color='blue', width=2, dash='dash')
    
    for c in range(1, 5):
        fig.add_trace(go.Scatter(x=empty_x, y=empty_y, mode='lines', line=line_r, name='Real' if c==1 else '', showlegend=(c==1)), row=3, col=c)
        fig.add_trace(go.Scatter(x=empty_x, y=empty_y, mode='lines', line=line_i, name='Imag' if c==1 else '', showlegend=(c==1)), row=3, col=c)

    # --- 4. LAYOUT ADJUSTMENTS ---
    fig.update_layout(
        height=1200, width=1600,
        title_text=f'Interactive Spectrum & Spatial Diagnostics | N = {bv_target}, m = {m_target}',
        title_font=dict(size=18, family="Arial", color="black"),
        template='plotly_white', hovermode='closest', margin=dict(t=80, b=40, l=40, r=40)
    )
    
    # Row 1 axes
    fig.update_xaxes(title_text="Scaled Wavenumber (k/N)", row=1, col=1)
    fig.update_xaxes(title_text="Scaled Wavenumber (k/N)", row=1, col=3)
    fig.update_yaxes(rangemode="tozero", row=1, col=1) 
    
    # Row 2 axes (Main Map)
    fig.update_xaxes(title_text="Radius (r)", range=[0, 4], row=2, col=1)
    fig.update_yaxes(title_text="Wave Frequency (\u03C3_i)", row=2, col=1)
    
    # Row 2 axes (Zoom Map)
    fig.update_xaxes(title_text="Radius (r)", range=[x_zmin, x_zmax], row=2, col=3)
    fig.update_yaxes(title_text="Wave Frequency (\u03C3_i)", range=[y_zmin, y_zmax], row=2, col=3)
    
    # Row 3 axes
    for i in range(1, 5): 
        fig.update_xaxes(title_text="Radius (r)", range=[0, r_max], row=3, col=i)

    f = go.FigureWidget(fig)

    # --- 5. THE CLICK CALLBACK ---
    def handle_click(trace, points, state):
        if not points.point_inds: return
        
        cdata = trace.customdata[points.point_inds[0]]
        base_dir, folder_name, mode_idx, k_raw, freq = cdata
        
        # Calculate dynamic S(r) bounds
        s_term = (m_target * bv_target) / np.sqrt(k_raw**2 * r_dense**2 + m_target**2)
        s_plus = -m_target * omega_base + s_term
        s_minus = -m_target * omega_base - s_term
        
        # Load spatial data
        vel_file = os.path.join(base_dir, 'bsnsq_vel', folder_name, f"ind_{mode_idx}.txt")
        has_data = False
        if os.path.exists(vel_file):
            try:
                data = np.loadtxt(vel_file, delimiter=',')
                r = data[:, 0]
                has_b = data.shape[1] >= 9
                has_data = True
            except: pass
            
        with f.batch_update():
            # Update Main Map Traces (Left)
            f.data[9].x = r_dense; f.data[9].y = s_plus
            f.data[10].x = r_dense; f.data[10].y = s_minus
            f.data[11].x = [0, r_max]; f.data[11].y = [freq, freq]

            # Update Zoom Map Traces (Right)
            f.data[17].x = r_dense; f.data[17].y = s_plus
            f.data[18].x = r_dense; f.data[18].y = s_minus
            f.data[19].x = [0, r_max]; f.data[19].y = [freq, freq]
            
            # Update Velocity Traces
            if has_data:
                for j in range(3):
                    f.data[20 + j*2].x = r; f.data[20 + j*2].y = data[:, 1 + j*2]
                    f.data[21 + j*2].x = r; f.data[21 + j*2].y = data[:, 2 + j*2]
                if has_b:
                    f.data[26].x = r; f.data[26].y = data[:, 7]
                    f.data[27].x = r; f.data[27].y = data[:, 8]
                else:
                    f.data[26].x = []; f.data[26].y = []; f.data[27].x = []; f.data[27].y = []
            
            k_scaled = k_raw / bv_target
            f.layout.title.text = f'Interactive Diagnostics | Mode ID: {mode_idx} | k/N: {k_scaled:.4f} | Freq: {freq:.4f}'

    # Bind click event to the 4 scatter plots in Row 1
    for i in range(4): f.data[i].on_click(handle_click)
    return f

In [ ]:
widget = create_interactive_full_dashboard(directory='../../data/l_10.0_n_1.0_nr_128', bv_target=1.0, m_target=1, r_max=20.0)
display(widget)

# Cross-check

In [ ]:
import os
import re
import glob
import numpy as np
import matplotlib.pyplot as plt

def parse_eigenvalue_file(filename):
    pattern = r"bsnsq_eig_bv_([+-]?\d+\.\d+)_w_([+-]?\d+\.\d+)_m_([+-]?\d+)_k_([+-]?\d+\.\d+)_nr_([+-]?\d+)\.(txt|output)"
    match = re.search(pattern, filename)
    if match:
        return {
            'filepath': filename, 'bv_freq': float(match.group(1)), 
            'm': int(match.group(3)), 'k': float(match.group(4)), 
            'eigenvalues': [], 'is_resolved': []
        }
    return None

def load_spectrum_dir(directory, bv_target=5.0, m_target=1):
    search_pattern = os.path.join(directory, f"bsnsq_eig_bv_{bv_target:.4f}_w_*_m_{m_target}_k_*.txt")
    files = glob.glob(search_pattern)
    if not files: files = glob.glob(search_pattern.replace('.txt', '.output'))
    
    data_dict = {}
    for f in files:
        d = parse_eigenvalue_file(f)
        if not d or not np.isclose(d['bv_freq'], bv_target) or d['m'] != m_target:
            continue
            
        k_val = d['k']
        with open(f, 'r') as file:
            for line in file:
                # Capture Real (Group 1), Imag (Group 2), and T/F Flag (Group 3)
                match = re.search(r'II\s*=\s*\d+\s*:\s*\(([^,]+),([^)]+)\)\s*-\s*([TF])', line)
                if match:
                    d['eigenvalues'].append(complex(float(match.group(1)), float(match.group(2))))
                    d['is_resolved'].append(match.group(3) == 'T')
                    
        # Store both eigenvalues and resolution status
        data_dict[k_val] = {
            'eigenvalues': np.array(d['eigenvalues']),
            'is_resolved': np.array(d['is_resolved'])
        }
    return data_dict

def plot_filtered_dispersion(dir_L1, dir_L2, bv_target=5.0, m_target=1):
    print(f"Loading L1 data from: {dir_L1}")
    data_L1 = load_spectrum_dir(dir_L1, bv_target, m_target)
    print(f"Loaded {len(data_L1)} k-values from L1.")
    
    print(f"Loading L2 data from: {dir_L2}")
    data_L2 = load_spectrum_dir(dir_L2, bv_target, m_target)
    print(f"Loaded {len(data_L2)} k-values from L2.")
    
    # Tolerances
    tol_freq = 2e-4      # Very strict on frequency
    tol_growth = 1e-3    # Loose on growth rate to allow for truncation error wobble
    
    filt_k, filt_re, filt_im = [], [], []
    
    # Floating-point robust matching for k values
    matched_k_pairs = []
    k2_array = np.array(list(data_L2.keys()))
    
    if len(k2_array) > 0:
        for k1 in data_L1.keys():
            diffs = np.abs(k2_array - k1)
            idx = np.argmin(diffs)
            if diffs[idx] < 1e-4:
                matched_k_pairs.append((k1, k2_array[idx]))
                
    matched_k_pairs.sort(key=lambda x: x[0])
    print(f"Found {len(matched_k_pairs)} overlapping 'k' values to cross-match.")
    
    for k1, k2 in matched_k_pairs:
        k_scaled = k1 / bv_target
        
        # Load properties from both directories
        eigs_1 = data_L1[k1]['eigenvalues']
        res_1  = data_L1[k1]['is_resolved']
        eigs_2 = data_L2[k2]['eigenvalues']
        
        for idx, e1 in enumerate(eigs_1):
            re1, im1 = np.real(e1), np.imag(e1)
            is_res = res_1[idx]
            
            # Condition 1: Mode is marked as resolved by the solver
            # Condition 2: Mode lies inside the Barotropic Critical Layer band
            in_CL = (-m_target - 1e-3 <= im1 <= 1e-3)
            
            if not (is_res or in_CL):
                continue
            
            # Skip decaying modes
            if re1 < 1e-6:
                continue
                
            # Find the minimum frequency distance to any mode in the L2 set
            freq_diffs = np.abs(np.imag(eigs_2) - im1)
            if len(freq_diffs) == 0:
                continue
                
            closest_idx = np.argmin(freq_diffs)
            best_e2 = eigs_2[closest_idx]
            re2, im2 = np.real(best_e2), np.imag(best_e2)
            
            # Apply the Rectangular Invariance Filter
            if abs(im1 - im2) < tol_freq and abs(re1 - re2) < tol_growth:
                filt_k.append(k_scaled)
                filt_re.append(re1)
                filt_im.append(im1)

    print(f"Filtering complete. Retained {len(filt_re)} invariant physical modes.")

    # --- Plotting ---
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    if filt_k:
        ax1.scatter(filt_k, filt_re, color='blue', s=2, label='Invariant Physical Modes')
        ax2.scatter(filt_k, filt_im, color='blue', s=2)
        
    ax1.axhline(0, color='black', linewidth=1, linestyle='--')
    ax1.set_xlabel(r'Scaled Wavenumber (k/N)')
    ax1.set_xlim(0, max(filt_k) * 1.05 if filt_k else 1)
    ax1.set_ylabel(r'Growth Rate ($\sigma_r$)')
    ax1.set_title(f'Filtered Growth Rates | N={bv_target}, m={m_target}')
    ax1.grid(True, linestyle=':', alpha=0.6)
    
    if filt_re: ax1.set_ylim(0, max(filt_re) * 1.1)

    ax2.axhline(0, color='red', linestyle='--', linewidth=1.5, label='Barotropic CL Bounds')
    ax2.axhline(-m_target, color='red', linestyle='--', linewidth=1.5)
    ax2.set_xlabel(r'Scaled Wavenumber (k/N)')
    ax2.set_xlim(0, max(filt_k) * 1.05 if filt_k else 1)
    ax2.set_ylabel(r'Wave Frequency ($\sigma_i$)')
    ax2.set_title('Filtered Frequencies')
    ax2.grid(True, linestyle=':', alpha=0.6)
    ax2.legend()
    
    plt.tight_layout()
    plt.show()

In [ ]:
plot_filtered_dispersion(dir_L1='../../data/l_6', dir_L2='../../data/l_5.5', bv_target=5.0, m_target=1)

# Production

In [ ]:
import os
import re
import glob
import numpy as np
import matplotlib.pyplot as plt

def parse_eigenvalue_file(filename):
    pattern = r"bsnsq_eig_bv_([+-]?\d+\.\d+)_w_([+-]?\d+\.\d+)_m_([+-]?\d+)_k_([+-]?\d+\.\d+)_nr_([+-]?\d+)\.(txt|output)"
    match = re.search(pattern, filename)
    if match:
        return {
            'filepath': filename, 'bv_freq': float(match.group(1)), 
            'm': int(match.group(3)), 'k': float(match.group(4)), 
            'eigenvalues': [], 'is_resolved': []
        }
    return None

def load_spectrum_dir(directory, bv_target=5.0, m_target=1):
    search_pattern = os.path.join(directory, f"bsnsq_eig_bv_{bv_target:.4f}_w_*_m_{m_target}_k_*.txt")
    files = glob.glob(search_pattern)
    if not files: files = glob.glob(search_pattern.replace('.txt', '.output'))
    
    data_dict = {}
    for f in files:
        d_meta = parse_eigenvalue_file(f)
        if not d_meta or not np.isclose(d_meta['bv_freq'], bv_target) or d_meta['m'] != m_target:
            continue
            
        k_key = round(d_meta['k'], 6)
        if k_key not in data_dict:
            data_dict[k_key] = {'eigenvalues': [], 'is_resolved': []}
            
        with open(f, 'r') as file:
            for line in file:
                match = re.search(r'II\s*=\s*\d+\s*:\s*\(([^,]+),([^)]+)\)\s*-\s*([TF])', line)
                if match:
                    val = complex(float(match.group(1)), float(match.group(2)))
                    res = (match.group(3) == 'T')
                    data_dict[k_key]['eigenvalues'].append(val)
                    data_dict[k_key]['is_resolved'].append(res)
    
    for k in data_dict:
        eigs = np.array(data_dict[k]['eigenvalues'])
        ress = np.array(data_dict[k]['is_resolved'])
        eigs_rounded = np.round(eigs.real, 12) + 1j * np.round(eigs.imag, 12)
        _, indices = np.unique(eigs_rounded, return_index=True)
        data_dict[k]['eigenvalues'] = eigs[indices]
        data_dict[k]['is_resolved'] = ress[indices]
    return data_dict

def split_branches(data):
    if len(data) == 0: return []
    branches = []
    current_branch = [data[0]]
    for i in range(1, len(data)):
        if data[i, 0] < data[i-1, 0] or (data[i, 0] - data[i-1, 0]) > 1.5:
            branches.append(np.array(current_branch))
            current_branch = []
        current_branch.append(data[i])
    if current_branch: branches.append(np.array(current_branch))
    return branches

def plot_validation_overlay(evp_dir, freq_csv, grow_csv, bv_target=5.0, m_target=1, 
                            tol_g=(0.04, 0.05, 0.06), tol_f=(0.01, 0.01, 0.01)):
    # 1. Load Theory Data
    freq_branches = split_branches(np.loadtxt(freq_csv, delimiter=','))
    grow_branches = split_branches(np.loadtxt(grow_csv, delimiter=','))
    num_branches = min(len(freq_branches), len(grow_branches))

    # 2. Load and Filter EVP pool
    evp_data = load_spectrum_dir(evp_dir, bv_target, m_target)
    bg_k, bg_re, bg_im, pool_by_k = [], [], [], {}

    for k_raw, d in evp_data.items():
        k_scaled = k_raw / bv_target
        pool_by_k[k_raw] = []
        for idx, e in enumerate(d['eigenvalues']):
            re_val, im_val = e.real, e.imag
            if re_val > 1e-6 and (d['is_resolved'][idx] or (-m_target - 1e-3 <= im_val <= 1e-3)):
                bg_k.append(k_scaled); bg_re.append(re_val); bg_im.append(im_val)
                pool_by_k[k_raw].append((re_val, im_val))

    # 3. Strict 1-to-1 Matching per k per branch
    matched_results = [([], [], []) for _ in range(num_branches)]
    for k_raw in sorted(pool_by_k.keys()):
        modes = pool_by_k[k_raw]
        if not modes: continue
        k_scaled = k_raw / bv_target
        re_arr = np.array([m[0] for m in modes])
        im_arr = np.array([m[1] for m in modes])
        claimed_indices = set()

        for b_idx in range(num_branches):
            b_f, b_g = freq_branches[b_idx], grow_branches[b_idx]
            if k_scaled < b_f[0,0] - 0.2 or k_scaled > b_f[-1,0] + 0.2: continue
            
            # Target values from theory
            target_f = -np.interp(k_scaled, b_f[:,0], b_f[:,1])
            target_g = np.interp(k_scaled, b_g[:,0], b_g[:,1])
            
            # Fetch specific tolerances for this branch
            curr_tol_g = tol_g[b_idx] if b_idx < len(tol_g) else 0.05
            curr_tol_f = tol_f[b_idx] if b_idx < len(tol_f) else 0.01
            
            # Distance components (Normalized)
            norm_f = max(np.max(b_f[:,1]), 1e-3)
            norm_g = max(np.max(b_g[:,1]), 1e-3)
            
            diff_g = np.abs(re_arr - target_g) / norm_g
            diff_f = np.abs(im_arr - target_f) / norm_f
            
            # Combined Euclidean distance for sorting the "best" candidate
            dist = np.sqrt(diff_g**2 + diff_f**2)
            
            # Mask modes already claimed or outside independent tolerances
            for c_idx in range(len(modes)):
                if c_idx in claimed_indices or diff_g[c_idx] > curr_tol_g or diff_f[c_idx] > curr_tol_f:
                    dist[c_idx] = np.inf
                
            best_idx = np.argmin(dist)
            if dist[best_idx] != np.inf:
                claimed_indices.add(best_idx)
                matched_results[b_idx][0].append(k_scaled)
                matched_results[b_idx][1].append(re_arr[best_idx])
                matched_results[b_idx][2].append(im_arr[best_idx])

    # 4. Plotting
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))
    colors, labels = ['tab:red', 'tab:green', 'tab:orange'], ['Ring (1st)', '1st mode', 'Ring (2nd)']
    
    for i in range(num_branches):
        m, b_g, b_f = matched_results[i], grow_branches[i], freq_branches[i]
        ax1.plot(b_g[:,0], b_g[:,1], color='black', lw=1.2, zorder=2)
        ax2.plot(b_f[:,0], -b_f[:,1], color='black', lw=1.2, zorder=2)
        if m[0]:
            ax1.scatter(m[0], m[1], edgecolors=colors[i], facecolors='none', s=50, lw=1.5, label=labels[i], zorder=3)
            ax2.scatter(m[0], m[2], edgecolors=colors[i], facecolors='none', s=50, lw=1.5, label=labels[i], zorder=3)

    ax1.set_xlabel(r'$k/N$'); ax1.set_ylabel(r'$\sigma_r$'); ax1.set_title('Growth Rate Validation')
    ax1.set_xlim(0, 35)
    ax1.set_ylim(0, max(bg_re) * 1.1 if bg_re else 1)
    ax2.set_xlabel(r'$k/N$'); ax2.set_ylabel(r'$\sigma_i$'); ax2.set_title('Frequency Validation')
    ax2.set_xlim(0, 35)
    ax2.set_ylim(-0.2, 0)
    ax1.legend(); ax2.legend(); plt.tight_layout(); plt.show()

# Execution example:
# plot_validation_overlay('l_6', 'n5_freq.csv', 'n5_grow.csv', 
#                         tol_g=(0.04, 0.1, 0.1), 
#                         tol_f=(0.01, 0.02, 0.02))

In [ ]:
plot_validation_overlay(
    evp_dir='../../data/l_6', 
    freq_csv='../../data/dizes/n5_freq.csv', 
    grow_csv='../../data/dizes/n5_grow.csv', 
    bv_target=5.0, 
    m_target=1,
    tol_g=(0.025, 0.025, 0.025),
    tol_f=(0.025, 0.15, 0.025)
)

In [ ]:
import os
import re
import glob
import numpy as np
import matplotlib.pyplot as plt

# Set global Matplotlib parameters for publication-quality fonts
plt.rcParams.update({
    'font.size': 18,          # Global font size
    'axes.labelsize': 20,     # X and Y axis labels
    'xtick.labelsize': 18,    # X tick numbers
    'ytick.labelsize': 18,    # Y tick numbers
    'legend.fontsize': 18,    # Legend text
    'axes.titlesize': 20,     # Subplot titles
    'text.usetex': True       # Set to True for LaTeX rendering
})

def parse_eigenvalue_file(filename):
    pattern = r"bsnsq_eig_bv_([+-]?\d+\.\d+)_w_([+-]?\d+\.\d+)_m_([+-]?\d+)_k_([+-]?\d+\.\d+)_nr_([+-]?\d+)\.(txt|output)"
    match = re.search(pattern, filename)
    if match:
        return {'filepath': filename, 'bv_freq': float(match.group(1)), 
                'm': int(match.group(3)), 'k': float(match.group(4))}
    return None

def load_spectrum_dir(directory, bv_target=5.0, m_target=1):
    search_pattern = os.path.join(directory, f"bsnsq_eig_bv_{bv_target:.4f}_w_*_m_{m_target}_k_*.txt")
    files = glob.glob(search_pattern)
    if not files: files = glob.glob(search_pattern.replace('.txt', '.output'))
    
    data_dict = {}
    for f in files:
        meta = parse_eigenvalue_file(f)
        if not meta or not np.isclose(meta['bv_freq'], bv_target) or meta['m'] != m_target:
            continue
        k_key = round(meta['k'], 6)
        if k_key not in data_dict:
            data_dict[k_key] = {'eigenvalues': [], 'is_resolved': []}
        with open(f, 'r') as file:
            for line in file:
                match = re.search(r'II\s*=\s*\d+\s*:\s*\(([^,]+),([^)]+)\)\s*-\s*([TF])', line)
                if match:
                    data_dict[k_key]['eigenvalues'].append(complex(float(match.group(1)), float(match.group(2))))
                    data_dict[k_key]['is_resolved'].append(match.group(3) == 'T')
    return data_dict

def split_and_sort_branches(data):
    if len(data) == 0: return []
    branches = []
    current_branch = [data[0]]
    for i in range(1, len(data)):
        if data[i, 0] < data[i-1, 0] or (data[i, 0] - data[i-1, 0]) > 1.5:
            branches.append(np.array(current_branch))
            current_branch = []
        current_branch.append(data[i])
    if current_branch: branches.append(np.array(current_branch))
    # CRITICAL: Sort by starting k to ensure Branch 0 = First Mode (0-7)
    branches.sort(key=lambda x: np.min(x[:, 0]))
    return branches

def plot_validation_overlay(evp_dir, freq_csv, grow_csv, bv_target=5.0, m_target=1, 
                            tol_g=(0.025, 0.025, 0.025), tol_f=(0.15, 0.025, 0.025)):
    # 1. Load and sort theoretical branches
    freq_branches = split_and_sort_branches(np.loadtxt(freq_csv, delimiter=','))
    grow_branches = split_and_sort_branches(np.loadtxt(grow_csv, delimiter=','))
    num_branches = min(len(freq_branches), len(grow_branches))

    # 2. Load EVP Data
    evp_data = load_spectrum_dir(evp_dir, bv_target, m_target)
    bg_k, bg_re, bg_im, pool_by_k = [], [], [], {}

    for k_raw, d in evp_data.items():
        k_scaled = k_raw / bv_target
        pool_by_k[k_raw] = []
        for idx, e in enumerate(d['eigenvalues']):
            re_val, im_val = e.real, e.imag
            if re_val > 1e-6 and (d['is_resolved'][idx] or (-m_target - 1e-3 <= im_val <= 1e-3)):
                bg_k.append(k_scaled); bg_re.append(re_val); bg_im.append(im_val)
                pool_by_k[k_raw].append((re_val, im_val))

    # 3. Match EVP Modes (1-to-1 matching using rectangular tolerance box)
    matched_results = [([], [], []) for _ in range(num_branches)]
    for k_raw in sorted(pool_by_k.keys()):
        modes = pool_by_k[k_raw]
        if not modes: continue
        k_scaled, re_arr, im_arr = k_raw / bv_target, np.array([m[0] for m in modes]), np.array([m[1] for m in modes])
        claimed_indices = set()

        for b_idx in range(num_branches):
            b_f, b_g = freq_branches[b_idx], grow_branches[b_idx]
            if k_scaled < b_f[0,0] - 0.2 or k_scaled > b_f[-1,0] + 0.2: continue
            
            target_f, target_g = -np.interp(k_scaled, b_f[:,0], b_f[:,1]), np.interp(k_scaled, b_g[:,0], b_g[:,1])
            norm_f, norm_g = max(np.max(b_f[:,1]), 1e-3), max(np.max(b_g[:,1]), 1e-3)
            
            # Independent error window check
            diff_g = np.abs(re_arr - target_g) / norm_g
            diff_f = np.abs(im_arr - target_f) / norm_f
            dist = np.sqrt(diff_g**2 + diff_f**2)
            
            for idx in range(len(modes)):
                # Mask modes already claimed or outside this branch's specific tolerance box
                if idx in claimed_indices or diff_g[idx] > tol_g[b_idx] or diff_f[idx] > tol_f[b_idx]:
                    dist[idx] = np.inf
                
            best_idx = np.argmin(dist)
            if dist[best_idx] != np.inf:
                claimed_indices.add(best_idx)
                matched_results[b_idx][0].append(k_scaled)
                matched_results[b_idx][1].append(re_arr[best_idx])
                matched_results[b_idx][2].append(im_arr[best_idx])

    # 4. Plotting
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
    colors, labels = ['tab:green', 'tab:red', 'tab:orange'], ['First mode', 'Ring mode (1st)', 'Ring mode (2nd)']
    
    # Label subfigures (a) and (b) at top left
    ax1.text(-0.15, 1.08, r'\textbf{(a)}', transform=ax1.transAxes, weight='bold')
    ax2.text(-0.15, 1.08, r'\textbf{(b)}', transform=ax2.transAxes, weight='bold')

    for i in range(num_branches):
        m, b_g, b_f = matched_results[i], grow_branches[i], freq_branches[i]
        # Theory lines
        ax1.plot(b_g[:,0], b_g[:,1], color='black', lw=1.5, zorder=2)
        ax2.plot(b_f[:,0], -b_f[:,1], color='black', lw=1.5, zorder=2)
        
        # Matched modes
        if m[0]:
            ax1.scatter(m[0], m[1], edgecolors=colors[i], facecolors='none', s=60, lw=1.5, label=labels[i], zorder=3)
            ax2.scatter(m[0], m[2], edgecolors=colors[i], facecolors='none', s=60, lw=1.5, label=labels[i], zorder=3)

    # Apply specific requested LaTeX labels
    ax1.set_xlabel(r'$k/\bar{N}$')
    ax1.set_ylabel(r'$\lambda$')
    
    ax2.set_xlabel(r'$k/\bar{N}$')
    ax2.set_ylabel(r'$\omega$')
    
    # ax1.legend(loc='upper right')
    ax2.legend(loc='lower right')
    
    ax1.set_xlim(0, 35)
    ax1.set_ylim(0, max(bg_re) * 1.1 if bg_re else 0.02)
    ax2.set_xlim(0, 35)
    ax2.set_ylim(-0.16, 0)
    
    plt.tight_layout()
    plt.savefig('validation_overlay.pdf', dpi=300, bbox_inches='tight')

    plt.show()

# Execution
plot_validation_overlay('../../data/l_6', '../../data/dizes/n5_freq.csv', '../../data/dizes/n5_grow.csv', 
                        tol_g=(0.025, 0.01, 0.01), 
                        tol_f=(0.1, 0.01, 0.01))

In [ ]:
import os
import re
import glob
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

def parse_eigenvalue_file(filename):
    pattern = r"bsnsq_eig_bv_([+-]?\d+\.\d+)_w_([+-]?\d+\.\d+)_m_([+-]?\d+)_k_([+-]?\d+\.\d+)_nr_([+-]?\d+)\.(txt|output)"
    match = re.search(pattern, filename)
    if match:
        return {
            'filepath': filename, 'bv_freq': float(match.group(1)), 'omega': float(match.group(2)),
            'm': int(match.group(3)), 'k': float(match.group(4)), 'nrchop': int(match.group(5)),
            'eigenvalues': [], 'is_resolved': [], 'indices': []
        }
    return None

def get_data(filename):
    data = parse_eigenvalue_file(filename)
    if not data: return None
    with open(filename, 'r') as file:
        for line in file:
            match = re.search(r'II\s*=\s*(\d+)\s*:\s*\(([^,]+),([^)]+)\)\s*-\s*([TF])', line)
            if match:
                data['indices'].append(int(match.group(1)))
                data['eigenvalues'].append(complex(float(match.group(2)), float(match.group(3))))
                data['is_resolved'].append(match.group(4) == 'T')
    data['indices'] = np.array(data['indices'])
    data['eigenvalues'] = np.array(data['eigenvalues'])
    data['is_resolved'] = np.array(data['is_resolved'])
    return data

def split_and_sort_branches(data):
    if len(data) == 0: return []
    branches = []
    current_branch = [data[0]]
    for i in range(1, len(data)):
        if data[i, 0] < data[i-1, 0] or (data[i, 0] - data[i-1, 0]) > 1.5:
            branches.append(np.array(current_branch))
            current_branch = []
        current_branch.append(data[i])
    if current_branch: branches.append(np.array(current_branch))
    branches.sort(key=lambda x: np.min(x[:, 0])) # Ensure Branch 0 is the First Mode
    return branches

def base_flow_profiles(r):
    with np.errstate(divide='ignore', invalid='ignore'):
        omega = (1 - np.exp(-r**2)) / r**2
        zeta = 2 * np.exp(-r**2)
    omega = np.where(r == 0, 1.0, omega)
    zeta = np.where(r == 0, 2.0, zeta)
    delta = 2 * omega * zeta
    return omega, delta

def create_interactive_full_dashboard(evp_dir='l_6', freq_csv='n5_freq.csv', grow_csv='n5_grow.csv', 
                                      bv_target=5.0, m_target=1, r_max=12.0,
                                      tol_g=(0.025, 0.025, 0.025), tol_f=(0.02, 0.02, 0.02)):
    
    # --- 1. THEORY DATA ---
    freq_branches = split_and_sort_branches(np.loadtxt(freq_csv, delimiter=','))
    grow_branches = split_and_sort_branches(np.loadtxt(grow_csv, delimiter=','))
    num_branches = min(len(freq_branches), len(grow_branches))

    # --- 2. EVP DATA POOLING & FILTERING ---
    search_pattern = os.path.join(evp_dir, f"bsnsq_eig_bv_{bv_target:.4f}_w_*_m_{m_target}_k_*.txt")
    files = glob.glob(search_pattern)
    if not files: files = glob.glob(search_pattern.replace('.txt', '.output'))
    if not files: return print("ERROR: No eigenvalue files found.")

    aggregated_data = [d for f in files if (d := get_data(f)) and np.isclose(d['bv_freq'], bv_target) and d['m'] == m_target]
    
    bg_k, bg_re, bg_im, bg_cdata = [], [], [], []
    pool_by_k = {}
    
    for d in aggregated_data:
        k_raw = d['k']
        k_scaled = k_raw / bv_target
        folder_name = os.path.basename(d['filepath']).replace('bsnsq_eig_', '').replace('.txt', '').replace('.output', '')
        
        # Bin to group identical k-values slightly offset by floating point
        k_bin = round(k_raw, 5)
        if k_bin not in pool_by_k: pool_by_k[k_bin] = []
        
        for i, e in enumerate(d['eigenvalues']):
            re_val, im_val = np.real(e), np.imag(e)
            is_res = d['is_resolved'][i]
            in_CL = (-m_target - 1e-3 <= im_val <= 1e-3)
            
            # Filter strictly unstable + physical bounds
            if re_val > 1e-6 and (is_res or in_CL):
                cdata = (evp_dir, folder_name, d['indices'][i], k_raw, im_val)
                bg_k.append(k_scaled); bg_re.append(re_val); bg_im.append(im_val); bg_cdata.append(cdata)
                
                pool_by_k[k_bin].append({
                    're': re_val, 'im': im_val, 'idx': d['indices'][i], 
                    'folder': folder_name, 'k_raw': k_raw, 'cdata': cdata
                })

    # --- 3. THEORY MATCHING (1-to-1) ---
    matched_results = [([], [], [], []) for _ in range(num_branches)] # k, re, im, cdata
    for k_bin in sorted(pool_by_k.keys()):
        modes = pool_by_k[k_bin]
        if not modes: continue
        
        k_scaled = modes[0]['k_raw'] / bv_target
        re_arr = np.array([m['re'] for m in modes])
        im_arr = np.array([m['im'] for m in modes])
        claimed_indices = set()

        for b_idx in range(num_branches):
            b_f, b_g = freq_branches[b_idx], grow_branches[b_idx]
            if k_scaled < b_f[0,0] - 0.2 or k_scaled > b_f[-1,0] + 0.2: continue
            
            target_f, target_g = -np.interp(k_scaled, b_f[:,0], b_f[:,1]), np.interp(k_scaled, b_g[:,0], b_g[:,1])
            norm_f, norm_g = max(np.max(b_f[:,1]), 1e-3), max(np.max(b_g[:,1]), 1e-3)
            
            diff_g = np.abs(re_arr - target_g) / norm_g
            diff_f = np.abs(im_arr - target_f) / norm_f
            dist = np.sqrt(diff_g**2 + diff_f**2)
            
            for idx in range(len(modes)):
                if idx in claimed_indices or diff_g[idx] > tol_g[b_idx] or diff_f[idx] > tol_f[b_idx]:
                    dist[idx] = np.inf
                
            best_idx = np.argmin(dist)
            if dist[best_idx] != np.inf:
                claimed_indices.add(best_idx)
                matched_results[b_idx][0].append(k_scaled)
                matched_results[b_idx][1].append(re_arr[best_idx])
                matched_results[b_idx][2].append(im_arr[best_idx])
                matched_results[b_idx][3].append(modes[best_idx]['cdata'])

    # --- 4. BASE FLOW PROFILES (STATIC) ---
    r_dense = np.linspace(0, r_max, 2000)
    omega_base, delta_base = base_flow_profiles(r_dense)
    tp_plus = -m_target * omega_base + np.sqrt(delta_base)
    tp_minus = -m_target * omega_base - np.sqrt(delta_base)
    barotropic_cl = -m_target * omega_base
    baroclinic_plus = -m_target * omega_base + bv_target
    baroclinic_minus = -m_target * omega_base - bv_target
    
    x_zmin, x_zmax = 1.0, r_max
    z_mask = (r_dense >= x_zmin) & (r_dense <= x_zmax)
    y_zmin, y_zmax = np.min(tp_plus[z_mask]) - 0.02, 0.05

    # --- 5. PLOTLY FIGURE SETUP ---
    fig = make_subplots(
        rows=3, cols=4, row_heights=[0.3, 0.45, 0.25],
        specs=[[{"colspan": 2}, None, {"colspan": 2}, None],
               [{"colspan": 2}, None, {"colspan": 2}, None],
               [{}, {}, {}, {}]],
        subplot_titles=("Growth Rate (\u03C3_r)", "Wave Frequency (\u03C3_i)", 
                        "Diagnostic Frequency Map", "Zoom: 3-Root Potential Well",
                        "u_r", "u_\u03B8", "u_z", "b (Density)"),
        vertical_spacing=0.1, horizontal_spacing=0.04
    )
    htemp = "k/N: %{x:.4f}<br>Value: %{y:.4e}<br>Mode ID: %{customdata[2]}<extra></extra>"

    # Row 1: EVP Cloud
    # fig.add_trace(go.Scatter(x=bg_k, y=bg_re, mode='markers', marker=dict(color='lightgray', size=4, opacity=0.4), name='Filtered Cloud', customdata=bg_cdata, hovertemplate=htemp), row=1, col=1)
    # fig.add_trace(go.Scatter(x=bg_k, y=bg_im, mode='markers', marker=dict(color='lightgray', size=4, opacity=0.4), showlegend=False, customdata=bg_cdata, hovertemplate=htemp), row=1, col=3)

    # Row 1: Theory & Matched Modes
    colors, labels = ['green', 'red', 'orange'], ['First Mode', '1st Ring', '2nd Ring']
    theory_added = False
    for i in range(num_branches):
        m, b_g, b_f = matched_results[i], grow_branches[i], freq_branches[i]
        
        fig.add_trace(go.Scatter(x=b_g[:,0].tolist(), y=b_g[:,1].tolist(), mode='lines', line=dict(color='black', width=1.2), name='Theory', showlegend=not theory_added, hoverinfo='skip'), row=1, col=1)
        fig.add_trace(go.Scatter(x=b_f[:,0].tolist(), y=(-b_f[:,1]).tolist(), mode='lines', line=dict(color='black', width=1.2), showlegend=False, hoverinfo='skip'), row=1, col=3)
        theory_added = True
        
        if m[0]:
            cdata_stack = m[3] # List of tuples
            m_style = dict(color='rgba(0,0,0,0)', size=7, line=dict(color=colors[i], width=2))
            fig.add_trace(go.Scatter(x=m[0], y=m[1], mode='markers', marker=m_style, name=labels[i], customdata=cdata_stack, hovertemplate=htemp), row=1, col=1)
            fig.add_trace(go.Scatter(x=m[0], y=m[2], mode='markers', marker=m_style, showlegend=False, customdata=cdata_stack, hovertemplate=htemp), row=1, col=3)

    # Row 2: Maps
    dash_p, dash_g, dash_br = dict(color='purple', dash='dash', width=2), dict(color='green', dash='dot', width=2), dict(color='saddlebrown', dash='dashdot', width=2)
    map_traces = [
        (tp_plus, dash_p, 'WKBJ Turning Pt (+)'), (tp_minus, dash_p, 'WKBJ Turning Pt (-)'),
        (barotropic_cl, dict(color='red', width=2), 'Barotropic CL'),
        (baroclinic_plus, dash_g, 'Baroclinic CL (+N)'), (baroclinic_minus, dash_g, 'Baroclinic CL (-N)')
    ]
    for y_data, style, name in map_traces:
        fig.add_trace(go.Scatter(x=r_dense, y=y_data, mode='lines', line=style, name=name), row=2, col=1)
        fig.add_trace(go.Scatter(x=r_dense, y=y_data, mode='lines', line=style, showlegend=False), row=2, col=3)

    # Dynamic Map Traces (Using unique names to easily target them in callback)
    dyn_map = [([], dash_br, 'S_p'), ([], dash_br, 'S_m'), ([], dict(color='black', width=2), 'F_line')]
    for y_data, style, name in dyn_map:
        fig.add_trace(go.Scatter(x=[], y=[], mode='lines', line=style, name=f'M_{name}', showlegend=False), row=2, col=1)
        fig.add_trace(go.Scatter(x=[], y=[], mode='lines', line=style, name=f'Z_{name}', showlegend=False), row=2, col=3)

    # Row 3: Velocities
    line_r, line_i = dict(color='black', width=2), dict(color='blue', width=2, dash='dash')
    v_names = ['ur', 'ut', 'uz', 'b']
    for c, v in enumerate(v_names, 1):
        fig.add_trace(go.Scatter(x=[], y=[], mode='lines', line=line_r, name=f'{v}_R', showlegend=(c==1)), row=3, col=c)
        fig.add_trace(go.Scatter(x=[], y=[], mode='lines', line=line_i, name=f'{v}_I', showlegend=(c==1)), row=3, col=c)

    # --- 6. LAYOUT & AXES ---
    fig.update_layout(height=1200, width=1600, title_text=f'Interactive Diagnostics | N={bv_target}, m={m_target}', template='plotly_white', hovermode='closest')
    
    max_re = max(bg_re) * 1.1 if bg_re else 0.16
    fig.update_xaxes(title_text="Scaled Wavenumber (k/N)", range=[0, 35], row=1, col=1)
    fig.update_xaxes(title_text="Scaled Wavenumber (k/N)", range=[0, 35], row=1, col=3)
    fig.update_yaxes(rangemode="tozero", range=[0, max_re], row=1, col=1)
    fig.update_yaxes(range=[-0.16, 0.02], row=1, col=3)
    fig.update_xaxes(title_text="Radius (r)", range=[0, 4], row=2, col=1)
    fig.update_yaxes(title_text="Wave Frequency (\u03C3_i)", row=2, col=1)
    fig.update_xaxes(title_text="Radius (r)", range=[x_zmin, x_zmax], row=2, col=3)
    fig.update_yaxes(title_text="Wave Frequency (\u03C3_i)", range=[y_zmin, y_zmax], row=2, col=3)
    for i in range(1, 5): fig.update_xaxes(title_text="Radius (r)", range=[0, r_max], row=3, col=i)

    f = go.FigureWidget(fig)

    # Map trace names to their absolute index for fast updates
    tr_idx = {tr.name: i for i, tr in enumerate(f.data) if tr.name}

    # --- 7. CALLBACK ---
    def handle_click(trace, points, state):
        if not points.point_inds: return
        
        cdata = trace.customdata[points.point_inds[0]]
        base_dir, folder_name, mode_idx, k_raw, freq = cdata
        
        # Calculate dynamic bounds
        s_term = (m_target * bv_target) / np.sqrt(k_raw**2 * r_dense**2 + m_target**2)
        s_plus = -m_target * omega_base + s_term
        s_minus = -m_target * omega_base - s_term
        
        vel_file = os.path.join(base_dir, 'bsnsq_vel', folder_name, f"ind_{mode_idx}.txt")
        has_data, r, v_data = False, [], []
        if os.path.exists(vel_file):
            try:
                v_data = np.loadtxt(vel_file, delimiter=',')
                r = v_data[:, 0]
                has_data = True
            except: pass
            
        with f.batch_update():
            # Update dynamic frequency lines
            for prefix in ['M_', 'Z_']:
                f.data[tr_idx[f'{prefix}S_p']].x = r_dense; f.data[tr_idx[f'{prefix}S_p']].y = s_plus
                f.data[tr_idx[f'{prefix}S_m']].x = r_dense; f.data[tr_idx[f'{prefix}S_m']].y = s_minus
                f.data[tr_idx[f'{prefix}F_line']].x = [0, r_max]; f.data[tr_idx[f'{prefix}F_line']].y = [freq, freq]
            
            # Update Velocity Traces
            if has_data:
                for j, v in enumerate(['ur', 'ut', 'uz']):
                    f.data[tr_idx[f'{v}_R']].x = r; f.data[tr_idx[f'{v}_R']].y = v_data[:, 1 + j*2]
                    f.data[tr_idx[f'{v}_I']].x = r; f.data[tr_idx[f'{v}_I']].y = v_data[:, 2 + j*2]
                if v_data.shape[1] >= 9:
                    f.data[tr_idx['b_R']].x = r; f.data[tr_idx['b_R']].y = v_data[:, 7]
                    f.data[tr_idx['b_I']].x = r; f.data[tr_idx['b_I']].y = v_data[:, 8]
            
            f.layout.title.text = f'Interactive Diagnostics | Mode: {mode_idx} | k/N: {k_raw/bv_target:.4f} | Freq: {freq:.4f}'

    # Bind click event to scatter plots (EVP Cloud + Matched Modes)
    for trace in f.data:
        if trace.mode == 'markers':
            trace.on_click(handle_click)
            
    return f


dashboard = create_interactive_full_dashboard(evp_dir='../../data/l_10.0_n_1.0_nr_128', freq_csv='../../data/dizes/n5_freq.csv', grow_csv='../../data/dizes/n5_grow.csv',
                                              tol_g=(0.025, 0.025, 0.025), tol_f=(0.1, 0.025, 0.025))
display(dashboard)

In [ ]:
import os
import re
import glob
import numpy as np
import matplotlib.pyplot as plt

# Set global Matplotlib parameters for publication-quality fonts
plt.rcParams.update({
    'font.size': 18,          
    'axes.labelsize': 20,     
    'xtick.labelsize': 18,    
    'ytick.labelsize': 18,    
    'legend.fontsize': 18,    
    'axes.titlesize': 20,     
    'text.usetex': True       
})

def parse_eigenvalue_file(filename):
    pattern = r"bsnsq_eig_bv_([+-]?\d+\.\d+)_w_([+-]?\d+\.\d+)_m_([+-]?\d+)_k_([+-]?\d+\.\d+)_nr_([+-]?\d+)\.(txt|output)"
    match = re.search(pattern, filename)
    if match:
        return {'filepath': filename, 'bv_freq': float(match.group(1)), 
                'm': int(match.group(3)), 'k': float(match.group(4))}
    return None

def load_spectrum_dir(directory, bv_target=5.0, m_target=1):
    search_pattern = os.path.join(directory, f"bsnsq_eig_bv_{bv_target:.4f}_w_*_m_{m_target}_k_*.txt")
    files = glob.glob(search_pattern)
    if not files: files = glob.glob(search_pattern.replace('.txt', '.output'))
    
    data_dict = {}
    for f in files:
        meta = parse_eigenvalue_file(f)
        if not meta or not np.isclose(meta['bv_freq'], bv_target) or meta['m'] != m_target:
            continue
        k_key = round(meta['k'], 6)
        if k_key not in data_dict:
            data_dict[k_key] = {'eigenvalues': [], 'is_resolved': [], 'indices': [], 'filepath': f}
        with open(f, 'r') as file:
            for line in file:
                # Added capture group (\d+) to properly capture the mode index
                match = re.search(r'II\s*=\s*(\d+)\s*:\s*\(([^,]+),([^)]+)\)\s*-\s*([TF])', line)
                if match:
                    data_dict[k_key]['indices'].append(int(match.group(1)))
                    data_dict[k_key]['eigenvalues'].append(complex(float(match.group(2)), float(match.group(3))))
                    data_dict[k_key]['is_resolved'].append(match.group(4) == 'T')
    return data_dict

def split_and_sort_branches(data):
    if len(data) == 0: return []
    branches = []
    current_branch = [data[0]]
    for i in range(1, len(data)):
        if data[i, 0] < data[i-1, 0] or (data[i, 0] - data[i-1, 0]) > 1.5:
            branches.append(np.array(current_branch))
            current_branch = []
        current_branch.append(data[i])
    if current_branch: branches.append(np.array(current_branch))
    branches.sort(key=lambda x: np.min(x[:, 0]))
    return branches

def base_flow_profiles(r, m_target=1):
    with np.errstate(divide='ignore', invalid='ignore'):
        omega = (1 - np.exp(-r**2)) / r**2
        zeta = 2 * np.exp(-r**2)
    omega = np.where(r == 0, 1.0, omega)
    zeta = np.where(r == 0, 2.0, zeta)
    delta = 2 * omega * zeta
    tp_plus = -m_target * omega + np.sqrt(delta)
    tp_minus = -m_target * omega - np.sqrt(delta)
    barotropic_cl = -m_target * omega
    return tp_plus, tp_minus, barotropic_cl

def generate_figures(evp_dir, freq_csv, grow_csv, bv_target=5.0, m_target=1, 
                     tol_g=(0.025, 0.025, 0.025), tol_f=(0.15, 0.025, 0.025)):
    # 1. Load and sort theoretical branches
    freq_branches = split_and_sort_branches(np.loadtxt(freq_csv, delimiter=','))
    grow_branches = split_and_sort_branches(np.loadtxt(grow_csv, delimiter=','))
    num_branches = min(len(freq_branches), len(grow_branches))

    # 2. Load EVP Data
    evp_data = load_spectrum_dir(evp_dir, bv_target, m_target)
    bg_k, bg_re, bg_im = [], [], []
    pool_by_k = {}

    for k_raw, d in evp_data.items():
        k_scaled = k_raw / bv_target
        pool_by_k[k_raw] = []
        for idx, e in enumerate(d['eigenvalues']):
            re_val, im_val = e.real, e.imag
            if re_val > 1e-6 and (d['is_resolved'][idx] or (-m_target - 1e-3 <= im_val <= 1e-3)):
                bg_k.append(k_scaled); bg_re.append(re_val); bg_im.append(im_val)
                pool_by_k[k_raw].append({
                    're': re_val, 'im': im_val, 
                    'idx': d['indices'][idx], 'filepath': d['filepath']
                })

    # 3. Match EVP Modes
    matched_results = [([], [], [], [], []) for _ in range(num_branches)] # k, re, im, idx, filepath
    for k_raw in sorted(pool_by_k.keys()):
        modes = pool_by_k[k_raw]
        if not modes: continue
        k_scaled = k_raw / bv_target
        re_arr = np.array([m['re'] for m in modes])
        im_arr = np.array([m['im'] for m in modes])
        claimed_indices = set()

        for b_idx in range(num_branches):
            b_f, b_g = freq_branches[b_idx], grow_branches[b_idx]
            if k_scaled < b_f[0,0] - 0.2 or k_scaled > b_f[-1,0] + 0.2: continue
            
            target_f, target_g = -np.interp(k_scaled, b_f[:,0], b_f[:,1]), np.interp(k_scaled, b_g[:,0], b_g[:,1])
            norm_f, norm_g = max(np.max(b_f[:,1]), 1e-3), max(np.max(b_g[:,1]), 1e-3)
            
            diff_g = np.abs(re_arr - target_g) / norm_g
            diff_f = np.abs(im_arr - target_f) / norm_f
            dist = np.sqrt(diff_g**2 + diff_f**2)
            
            for idx in range(len(modes)):
                if idx in claimed_indices or diff_g[idx] > tol_g[b_idx] or diff_f[idx] > tol_f[b_idx]:
                    dist[idx] = np.inf
                
            best_idx = np.argmin(dist)
            if dist[best_idx] != np.inf:
                claimed_indices.add(best_idx)
                matched_results[b_idx][0].append(k_scaled)
                matched_results[b_idx][1].append(re_arr[best_idx])
                matched_results[b_idx][2].append(im_arr[best_idx])
                matched_results[b_idx][3].append(modes[best_idx]['idx'])
                matched_results[b_idx][4].append(modes[best_idx]['filepath'])

    # Target points on the FIRST branch
    target_ks = [1.4, 2.4, 4.8]
    targets = []
    b0_k = np.array(matched_results[0][0])
    for tk in target_ks:
        best_i = np.argmin(np.abs(b0_k - tk))
        targets.append({
            'k': b0_k[best_i], 're': matched_results[0][1][best_i], 'im': matched_results[0][2][best_i],
            'idx': matched_results[0][3][best_i], 'filepath': matched_results[0][4][best_i]
        })

    # =========================================================================
    # FIGURE 1: Validation Overlay
    # =========================================================================
    fig1, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
    colors, labels = ['tab:green', 'tab:red', 'tab:orange'], ['First mode', 'Ring mode (1st)', 'Ring mode (2nd)']
    
    ax1.text(-0.12, 1.05, r'\textbf{(a)}', transform=ax1.transAxes, weight='bold')
    ax2.text(-0.12, 1.05, r'\textbf{(b)}', transform=ax2.transAxes, weight='bold')

    # Plot base branches
    for i in range(num_branches):
        m, b_g, b_f = matched_results[i], grow_branches[i], freq_branches[i]
        
        ax1.plot(b_g[:,0], b_g[:,1], color='black', lw=1.5, zorder=2)
        ax2.plot(b_f[:,0], -b_f[:,1], color='black', lw=1.5, zorder=2)
        
        if m[0]:
            # Left panel: No label for the branches (per requirements)
            ax1.scatter(m[0], m[1], edgecolors=colors[i], facecolors='none', s=60, lw=1.5, zorder=3)
            # Right panel: Add branch labels for legend
            ax2.scatter(m[0], m[2], edgecolors=colors[i], facecolors='none', s=60, lw=1.5, label=labels[i], zorder=3)

    # Plot the 3 specific target points on BOTH panels
    markers = ['s', '^', 'D'] # Square, Triangle, Diamond
    for i, tgt in enumerate(targets):
        # Left Panel gets the legend labels for the symbols
        ax1.scatter(tgt['k'], tgt['re'], color='tab:green', marker=markers[i], s=120, zorder=4, label=rf'$k/\bar{{N}} = {tgt["k"]:.1f}$')
        # Right Panel gets symbols plotted, but NO label
        ax2.scatter(tgt['k'], tgt['im'], color='tab:green', marker=markers[i], s=120, zorder=4)

    ax1.set_xlabel(r'$k/\bar{N}$')
    ax1.set_ylabel(r'$\lambda$')
    ax2.set_xlabel(r'$k/\bar{N}$')
    ax2.set_ylabel(r'$\omega$')
    
    # Legends strictly separated as requested
    ax1.legend(loc='upper right', frameon=False)
    ax2.legend(loc='lower right', frameon=False)
    
    ax1.set_xlim(0, 35)
    ax1.set_ylim(0, max(bg_re) * 1.1 if bg_re else 0.02)
    ax2.set_xlim(0, 35)
    ax2.set_ylim(-0.16, 0.005)
    
    fig1.tight_layout()
    fig1.savefig('validation_overlay.pdf', dpi=300, bbox_inches='tight')
    print("Figure 1 saved as validation_overlay.pdf")

    # =========================================================================
    # FIGURE 2: Mode Structure (3 Rows: Map & U_theta)
    # =========================================================================
    fig2, axes2 = plt.subplots(3, 2, figsize=(12, 9))
    row_labels = [r'\textbf{(a)}', r'\textbf{(b)}', r'\textbf{(c)}']
    
    r_dense = np.linspace(0, 5, 1000)
    tp_plus, tp_minus, barotropic_cl = base_flow_profiles(r_dense, m_target)

    for i, tgt in enumerate(targets):
        ax_map = axes2[i, 0]
        ax_vel = axes2[i, 1]
        
        # Subfigure label top left of each row
        ax_map.text(-0.17, 1.08, row_labels[i], transform=ax_map.transAxes, weight='bold')

        # --- 1. Frequency Map ---
        ax_map.plot(r_dense, tp_plus, 'm--', lw=1.5, label=r'$-m\bar{\Omega}+\omega^{\pm}_H$' if i==0 else "")
        ax_map.plot(r_dense, tp_minus, 'm--', lw=1.5)
        ax_map.plot(r_dense, barotropic_cl, 'r-', lw=1.5, label=r'$-m\bar{\Omega}$' if i==0 else "")
        ax_map.axhline(tgt['im'], color='tab:green', linestyle='-', lw=2, label=r'$\omega$' if i==0 else "")
        
        # Mark specific marker on the frequency line
        ax_map.scatter([0.2], [tgt['im']], color='tab:green', marker=markers[i], s=100, zorder=5)

        ax_map.set_xlim(0, 4)
        ax_map.set_ylim(-0.25, 0.05)
        ax_map.set_ylabel(r'$\omega$')
        
        if i == 0: ax_map.legend(loc='upper right', fontsize=12, frameon=False)
        if i == 2: ax_map.set_xlabel(r'Radius $r$')
        else: ax_map.set_xticklabels([])

        # --- 2. U_theta Profile ---
        base_name = os.path.basename(tgt['filepath'])
        folder_name = base_name.replace('bsnsq_eig_', '').replace('.txt', '').replace('.output', '')
        vel_file = os.path.join(evp_dir, 'bsnsq_vel', folder_name, f"ind_{tgt['idx']}.txt")
        
        if os.path.exists(vel_file):
            v_data = np.loadtxt(vel_file, delimiter=',')
            r_vel = v_data[:, 0]
            ut_R, ut_I = v_data[:, 3], v_data[:, 4]
            
            # Normalize
            norm_factor = np.max(np.abs(ut_R)) if np.max(np.abs(ut_R)) > 0 else 1.0
            
            ax_vel.plot(r_vel, ut_R / norm_factor, 'k-', lw=1.5, label='Real' if i==0 else "")
            ax_vel.plot(r_vel, ut_I / norm_factor, 'k--', lw=1.5, label='Imag' if i==0 else "")
        else:
            ax_vel.text(0.5, 0.5, "Velocity data missing", ha='center', va='center', transform=ax_vel.transAxes)

        ax_vel.set_xlim(0, 4)
        ax_vel.set_ylabel(r'$\tilde{u}_\theta$')
        
        # Identify the mode context in the plot
        ax_vel.text(0.95, 0.85, rf'$k/\bar{{N}}={tgt["k"]:.1f}$', transform=ax_vel.transAxes, 
                    ha='right', va='top', color='tab:green', weight='bold')

        if i == 0: ax_vel.legend(loc='lower left', fontsize=12, frameon=False)
        if i == 2: ax_vel.set_xlabel(r'Radius $r$')
        else: ax_vel.set_xticklabels([])

    fig2.tight_layout()
    fig2.savefig('mode_structure.pdf', dpi=300, bbox_inches='tight')
    print("Figure 2 saved as mode_structure.pdf")


# Execute
generate_figures('../../data/l_6', '../../data/dizes/n5_freq.csv', '../../data/dizes/n5_grow.csv', 
                 tol_g=(0.025, 0.025, 0.025), tol_f=(0.1, 0.025, 0.025))